# CymbalGoal AlloyDB Workshop—Scout Reports: Generate, Embed, Stage

**Track A, pipeline step 01.** Builds the scouting-report corpus the workshop labs search over:
one report per scouting club per player-season, for every player-season with **900+ minutes**,
written by up to **two clubs** with genuinely different viewpoints. The 00 data spike sized it
at **38,445 reports**.

| | |
| :-- | :-- |
| **Reads** | the CymbalGoal tables at `gs://class-demo/alloydb-labs/cymbalgoal/` (read-only, another team's labs use it) |
| **Writes** | only under `gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop/scout-reports/` |
| **Generates** | `gemini-3.8-flash`, no Search grounding, with Part 2's threaded, throttled, checkpointed loop |
| **Embeds** | `gemini-embedding-001` at 3072 dims (locked by Lab 1's in-SQL `embedding()`), online, resumable |
| **Stages** | headerless `scout_reports*.csv.gz` (full corpus and one file per season), `scout_reports.sql`, a manifest |

### The three runs

Run all is always safe. Every step that costs money is behind a flag, and every finished step
is saved to the bucket so it is never redone.

| Run | Set in section 1 | What happens |
| :-- | :-- | :-- |
| **1** | nothing (defaults) | Builds all fact blocks, generates the **25-report sample** (pennies), runs QA on it. **Stop and review.** |
| **2** | `RUN_FULL = True` | Generates all reports with Part 2's loop, checkpointing as it goes; then QA and repair, embeddings, staging. |
| **2+** | leave `RUN_FULL = True` | If the runtime dies or you stop the cell, Run all again: it resumes from the checkpoint. Repeat until section 13 says **DONE**. |

The last cell, **14 · Handback**, shows one box with everything Claude needs and a **Copy** button.

### How a report is written

- **As of the end of that season.** The scout knows nothing that happened later. The "later
  signer" is the club that eventually buys him, but the report never mentions that purchase.
  The relationship lives in the `perspective` column, not in the text.
- **Facts only.** Every number in a report comes from the fact block built here from the tables.
  Gaps in appearances are described as absences, never as injuries, because the data does not
  say why.
- **Disambiguated.** Each report opens with the player's full name, club and age, so the six
  Paulinhos can't be confused with each other.

---
## 1 · Configuration

The run flags are at the top. Everything else is a knob with a reason next to it.

In [ ]:
# ===== RUN FLAGS—the only things you normally change ======================================
RUN_SAMPLE       = True    # run 1: 25 reports online (cached in the bucket after the first run)
RERUN_SAMPLE     = False   # True = regenerate the sample even though a cached one exists
RUN_FULL         = False   # run 2: generate every report (resumes from the checkpoint on reruns)
REPAIR_ONLINE    = True    # after generation: regenerate reports that fail hard QA
RUN_EMBEDDINGS   = True    # after QA: embed every report (resumable shards)
STAGE_EXPORT     = True    # after embeddings: write and upload the staged CSVs (once)
FORCE_RESTAGE    = False   # True = rewrite the staged files even though they already match
# =============================================================================================

In [ ]:
# google-genai must be recent enough for thinking_level (Gemini 3.x).
import os, subprocess, sys
if not os.environ.get("CG_FIXTURE"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "google-genai"], check=True)
print("google-genai ready")

In [ ]:
import csv, gzip, hashlib, io, json, math, os, random, re, subprocess, sys, threading, time
import unicodedata
from collections import Counter, defaultdict
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, as_completed
from concurrent.futures import wait as fut_wait
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

# ---- Provenance ----------------------------------------------------------------------------
SNAPSHOT_DATE   = "2026-08-14"
SNAPSHOT_SHA256 = "3e6742a9c5002b202eeb8803a123bbc2d9c13ae582eb45ffb6f587843693fffa"
PROMPT_VERSION  = "sr-v1.3.2"    # bump on ANY prompt or fact-block change; names every artifact

# ---- Where things live ---------------------------------------------------------------------
GCS_READ = "gs://class-demo/alloydb-labs/cymbalgoal"                                  # read-only
GCS_WORK = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop/scout-reports"   # ours
BASE = Path("/content/cymbalgoal-scout-reports")
try:
    BASE.mkdir(parents=True, exist_ok=True)
except OSError:
    BASE = Path.home() / "cymbalgoal-scout-reports"
    BASE.mkdir(parents=True, exist_ok=True)
for _d in ("results", "emb", "export"):
    (BASE / _d).mkdir(exist_ok=True)

PROJECT_ID = None                # None = detect from the runtime

# ---- Models --------------------------------------------------------------------------------
GEN_MODEL      = "gemini-3.8-flash"
GEN_LOCATIONS  = ["global", "us-central1"]    # Gemini 3.x is served from global (Part 2)
EMBED_MODEL    = "gemini-embedding-001"
EMBED_DIM      = 3072            # LOCKED: Lab 1 embeds queries in SQL at the model's native width
EMBED_TASK_TYPE = "RETRIEVAL_DOCUMENT"
EMBED_LOCATIONS = ["us-central1", "us-east1", "europe-west4"]   # regional only (Part 2)
EMBED_SIG_DIGITS = 6
TEMPERATURE    = 0.7             # two scouts on one player-season should not read identically
SEED           = 2026
THINKING_LEVEL = "LOW"           # no "off" on Gemini 3.x; reasoning bills at the output rate
MAX_OUTPUT_TOKENS = 4096

# ---- Corpus definition (agreed after the 00 data spike) -----------------------------------
BIG5 = ["GB1", "ES1", "IT1", "L1", "FR1"]
MIN_MINUTES   = 900
SCOUTS_PER_PS = 2
EXPECTED_REPORTS = 38_445        # what 00 measured; a mismatch is reported, not fatal
REPORT_WORDS  = (120, 180)
HARD_WORDS    = (90, 230)        # outside this, a report is regenerated
SAMPLE_N      = 25
PRIMARY_ABSENCE = (6, 30)        # >= 6 consecutive club matches AND >= 30 days (00 definition)
ABSENCE_MIN_APPS = 5
TRAILING_ABSENCE_MIN = 6

# ---- Throughput and resilience: Part 2's values ---------------------------------------------
MAX_WORKERS = 50                 # concurrent calls to start with
MAX_WORKERS_CEILING = 150        # the throttle may grow past MAX_WORKERS while nothing is rate-limited
MIN_WORKERS = 4
THROTTLE_WINDOW_S = 30.0         # the throttle looks at the last window of calls this often
SAVE_EVERY_N_REPORTS = 100       # a count of reports finished in this run, never len(results) % N
GCS_PUSH_EVERY_S = 120.0         # mirror the checkpoint to the bucket at most every 2 minutes
THROTTLE_TRIGGER_RATE = 0.15     # > 15% of a window's calls rate-limited -> shrink the pool to 60%
THROTTLE_GROW_BELOW = 0.03       # < 3% rate-limited -> grow the pool (a stray 429 is no reason to hold back)
MAX_RETRIES = 8
REQUEST_TIMEOUT_S = 90.0         # per call; a slower call is abandoned and retried
RETRY_BUDGET_S = 120.0           # per report, across all attempts
BACKOFF_BASE = 2.0
BACKOFF_CAP_S = 60.0
RATE_LIMIT_MIN_WAIT_S = 20.0     # 429s are outwaited, never hammered
SAMPLE_WORKERS = 8
EMBED_WORKERS = 32
EMBED_SHARD = 2000               # rows per embedding shard file (resume unit)
REPAIR_ROUNDS = 3
REPAIR_LIMIT = 4000              # more hard failures than this is a prompt problem, not bad luck

# ---- Pricing (USD per 1M tokens), arithmetic only ------------------------------------------
PRICE_IN_PER_M, PRICE_OUT_PER_M = 0.75, 3.75     # promotional list price, online (no batch discount)
PRICE_EMBED_PER_M = 0.15

TABLES = ["competitions", "clubs", "players", "games", "appearances", "game_events", "transfers",
          "player_valuations"]
EXPECTED_ROWS = {"competitions": 65, "clubs": 796, "players": 13_439, "games": 29_740,
                 "appearances": 832_193, "game_events": 417_617, "transfers": 65_494,
                 "player_valuations": 297_822}

print(f"prompt {PROMPT_VERSION}   model {GEN_MODEL}   embed {EMBED_MODEL}@{EMBED_DIM}")
print(f"reads  {GCS_READ}/ (read-only)\nwrites {GCS_WORK}/")
print(f"flags  sample={RUN_SAMPLE} full={RUN_FULL} repair={REPAIR_ONLINE} "
      f"embed={RUN_EMBEDDINGS} stage={STAGE_EXPORT}")

### 1.1 The bucket guard

Every bucket call goes through `gcs()`. Reads are allowed under either prefix. Writes (`cp`)
are allowed **only** under the workshop prefix, and nothing can delete. The two prefixes share a
stem (`cymbalgoal` / `cymbalgoal-alloydb-workshop`), so the check matches on a trailing `/`,
never on a bare string prefix.

`CG_FIXTURE` is an offline test hook; it is unset in Colab.

In [ ]:
GCS_LOG = []
_FIXTURE = os.environ.get("CG_FIXTURE")


def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")


def _guard(args):
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if verb in ("ls", "cat"):
        bad = [u for u in uris if not (_under(u, GCS_READ) or _under(u, GCS_WORK))]
    elif verb == "cp":
        dest = args[-1]
        if not _under(dest, GCS_WORK):
            raise PermissionError(f"gcs(): writes are only allowed under {GCS_WORK}/—refused {dest}")
        bad = [u for u in uris[:-1] if not (_under(u, GCS_READ) or _under(u, GCS_WORK))]
    else:
        raise PermissionError(f"gcs(): '{verb}' refused—only ls, cat and cp are allowed")
    if bad:
        raise PermissionError(f"gcs(): outside the allowed prefixes: {bad}")
    GCS_LOG.append("gcloud storage " + " ".join(args))


def _fx(uri):
    return Path(_FIXTURE) / uri[len("gs://"):]


def _fixture_run(args):
    import shutil
    verb = args[0]
    if verb == "ls":
        uri = [a for a in args if a.startswith("gs://")][0]
        root = _fx(uri)
        if not root.exists():
            return subprocess.CompletedProcess(args, 1, b"", b"matched no objects")
        files = [root] if root.is_file() else (sorted(root.rglob("*")) if "-r" in args else sorted(root.iterdir()))
        lines = [f"{f.stat().st_size:>10}  2026-10-13T00:00:00Z  gs://{f.relative_to(Path(_FIXTURE)).as_posix()}"
                 for f in files if f.is_file()]
        return subprocess.CompletedProcess(args, 0, ("\n".join(lines) + "\n").encode(), b"")
    if verb == "cat":
        p = _fx(args[-1])
        return (subprocess.CompletedProcess(args, 0, p.read_bytes(), b"") if p.exists()
                else subprocess.CompletedProcess(args, 1, b"", b"No URLs matched"))
    if verb == "cp":
        dest = args[-1]
        for src in args[1:-1]:
            if src.startswith("-"):
                continue
            d = _fx(dest)
            if dest.endswith("/"):
                d = d / Path(src).name
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(_fx(src) if src.startswith("gs://") else src, d)
        return subprocess.CompletedProcess(args, 0, b"", b"")


def gcs(*args, binary=False):
    _guard(args)
    r = _fixture_run(args) if _FIXTURE else subprocess.run(["gcloud", "storage", *args], capture_output=True)
    if not binary:
        r = subprocess.CompletedProcess(r.args, r.returncode, r.stdout.decode("utf-8", "replace"),
                                        r.stderr.decode("utf-8", "replace"))
    return r


def gcs_stream(uri):
    _guard(("cat", uri))
    if _FIXTURE:
        fh = open(_fx(uri), "rb")
        return fh, fh.close
    p = subprocess.Popen(["gcloud", "storage", "cat", uri], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)

    def _close():
        p.stdout.close(); p.kill(); p.wait()
    return p.stdout, _close


_LS_RE = re.compile(r"^\s*(\d+)\s+(\S+)\s+(gs://\S+)\s*$")


def ls(uri, recursive=True):
    r = gcs("ls", "-l", *(["-r"] if recursive else []), uri.rstrip("/") + "/")
    out = {}
    if r.returncode == 0:
        for line in r.stdout.splitlines():
            m = _LS_RE.match(line)
            if m:
                out[m.group(3)] = int(m.group(1))
    return out


def put(local, uri):
    r = gcs("cp", str(local), uri)
    if r.returncode != 0:
        raise RuntimeError(f"upload to {uri} failed: {r.stderr.strip()[:400]}")


def get_text(uri):
    r = gcs("cat", uri)
    return r.stdout if r.returncode == 0 else None


# ---- handback -----------------------------------------------------------------------------
HB = {}
TIMERS = {}


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def show(df, index=False, max_rows=60):
    print(df.to_string(index=index, max_rows=max_rows))


def money(v):
    if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
        return "unknown"
    v = float(v)
    if v == 0:
        return "free"
    if v >= 1e6:
        return f"€{v / 1e6:,.1f}m".replace(".0m", "m")
    return f"€{v / 1e3:,.0f}k"


def season_label(s):
    return f"{s}/{str(s + 1)[-2:]}"


def fold(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(ch)).lower()


print("guard armed" + ("  [OFFLINE FIXTURE MODE]" if _FIXTURE else ""))

---
## 2 · Preflight

Nothing below this runs until the preflight passes. It resolves the project, proves the runtime
can **write** to the workshop prefix (one tiny marker object), and finds a location that serves
each model. The two probe calls cost a fraction of a cent.

If the write probe fails, the Qwiklabs project's service account needs write access on the
`class-demo` bucket (Storage Object Admin is enough), or at least on the workshop prefix.

In [ ]:
S = "2 · Preflight"
from google import genai
from google.genai import types as gtypes

if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.run(
        ["gcloud", "config", "get-value", "project"], capture_output=True, text=True).stdout.strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    raise RuntimeError("Could not detect the project. Set PROJECT_ID in section 1.")

# Which identity the runtime's gcloud calls run as: that is the principal that needs the grant.
ACCOUNT = "fixture" if _FIXTURE else subprocess.run(
    ["gcloud", "config", "get-value", "account"], capture_output=True, text=True).stdout.strip() or "unknown"
PRINCIPAL = (f"serviceAccount:{ACCOUNT}" if ACCOUNT.endswith("gserviceaccount.com") else f"user:{ACCOUNT}")
GRANT_CMD = (f'gcloud storage buckets add-iam-policy-binding gs://class-demo --member="{PRINCIPAL}" '
             f'--role="roles/storage.objectAdmin"')
print(f"runtime identity: {ACCOUNT}")
_probe = BASE / "_write_probe.txt"
_probe.write_text(f"write probe {datetime.now(timezone.utc).isoformat()} from {PROJECT_ID}\n")
WRITE_ERR = ""
try:
    put(_probe, f"{GCS_WORK}/_write_probe.txt")
    WRITE_OK = True
    print(f"✅ can write to {GCS_WORK}/")
except RuntimeError as exc:
    WRITE_OK = False
    WRITE_ERR = " ".join(str(exc).split())[:400]
    print(f"🔴 cannot write to {GCS_WORK}/\n   {WRITE_ERR}\n\n"
          f"   Grant {PRINCIPAL} write access, from Cloud Shell as the bucket owner:\n\n   {GRANT_CMD}\n\n"
          "   Then rerun. Nothing else is needed: the read prefix already works.")

_tls = threading.local()


def client(location):
    cache = getattr(_tls, "c", None)
    if cache is None:
        cache = _tls.c = {}
    if location not in cache:
        try:                    # a per-request timeout, so a hung call is retried instead of waited on forever
            cache[location] = genai.Client(vertexai=True, project=PROJECT_ID, location=location,
                                           http_options=gtypes.HttpOptions(timeout=int(REQUEST_TIMEOUT_S * 1000)))
        except (TypeError, AttributeError):
            cache[location] = genai.Client(vertexai=True, project=PROJECT_ID, location=location)
    return cache[location]


_GCC = set(getattr(gtypes.GenerateContentConfig, "model_fields", None) or {})


def thinking_config():
    try:
        return gtypes.ThinkingConfig(thinking_level=THINKING_LEVEL)
    except Exception:
        print("⚠️  this google-genai build rejects thinking_level; upgrade the SDK")
        return None


def afc_off():
    if "automatic_function_calling" not in _GCC:
        return {}
    try:
        return {"automatic_function_calling": gtypes.AutomaticFunctionCallingConfig(disable=True)}
    except Exception:
        return {}


GEN_LOCATION = EMBED_LOCATION = None
for loc in GEN_LOCATIONS:
    try:
        r = client(loc).models.generate_content(
            model=GEN_MODEL, contents="Reply with the single word: ready",
            config=gtypes.GenerateContentConfig(temperature=0, max_output_tokens=512, **afc_off()))
        GEN_LOCATION = loc
        print(f"✅ {GEN_MODEL} @ {loc}: {(r.text or '').strip()[:20]!r}")
        break
    except Exception as exc:  # noqa: BLE001
        print(f"   {GEN_MODEL} @ {loc}: {type(exc).__name__}: {str(exc)[:160]}")
for loc in EMBED_LOCATIONS:
    try:
        r = client(loc).models.embed_content(model=EMBED_MODEL, contents="ready",
                                             config=gtypes.EmbedContentConfig(output_dimensionality=EMBED_DIM))
        n = len(r.embeddings[0].values)
        if n == EMBED_DIM:
            EMBED_LOCATION = loc
            print(f"✅ {EMBED_MODEL} @ {loc}: {n} dims")
            break
        print(f"   {EMBED_MODEL} @ {loc}: {n} dims (need {EMBED_DIM})")
    except Exception as exc:  # noqa: BLE001
        print(f"   {EMBED_MODEL} @ {loc}: {type(exc).__name__}: {str(exc)[:160]}")

PREFLIGHT_OK = bool(WRITE_OK and GEN_LOCATION and EMBED_LOCATION)
print("\n" + ("✅ PREFLIGHT PASSED" if PREFLIGHT_OK else "🔴 PREFLIGHT FAILED—later sections will skip"))
hb(S, f"project {PROJECT_ID}; identity {ACCOUNT}; write {'ok' if WRITE_OK else 'FAILED'}; gen {GEN_MODEL}@{GEN_LOCATION}; "
      f"embed {EMBED_MODEL}@{EMBED_LOCATION}; google-genai {getattr(genai, '__version__', '?')}")
if not WRITE_OK:
    hb(S, f"write error: {WRITE_ERR}")
    hb(S, f"grant: {GRANT_CMD}")

---
## 3 · Load the tables

Same loader as 00: the staged headerless CSVs from the read-only prefix, column order from
`manifest.json`, dtypes forced the Part 2 way. `player_valuations` is added because each report
carries the player's market value at the start and end of the season.

In [ ]:
S = "3 · Load the tables"
INT_COLS = {"club_id", "player_id", "game_id", "transfer_id", "current_club_id", "player_club_id",
            "home_club_id", "away_club_id", "from_club_id", "to_club_id", "season", "last_season",
            "height_in_cm", "market_value_in_eur", "highest_market_value_in_eur", "transfer_fee",
            "home_club_goals", "away_club_goals", "home_club_position", "away_club_position",
            "yellow_cards", "red_cards", "goals", "assists", "minutes_played", "total_clubs",
            "international_caps", "international_goals", "minute", "player_in_id", "player_assist_id"}
DATE_COLS = {"date_of_birth", "game_date", "appearance_date", "valuation_date", "transfer_date", "event_date",
             "contract_expiration_date"}
TEXT_COLS = {"appearance_id", "competition_id", "domestic_competition_id", "round_label",
             "from_club_name", "to_club_name", "player_name", "club_name", "competition_name",
             "transfer_season", "foot", "main_position", "detailed_position", "country_of_citizenship",
             "game_event_id", "event_type", "description", "home_club_manager_name", "away_club_manager_name",
             "stadium", "city_of_birth", "country_of_birth"}


def normalize(df):
    for c in df.columns:
        if c in INT_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
        elif c in DATE_COLS:
            df[c] = pd.to_datetime(df[c], errors="coerce")
        elif c in TEXT_COLS:
            df[c] = df[c].astype("string")
    return df


MANIFEST = json.loads(get_text(f"{GCS_READ}/manifest.json") or "{}")
t0 = time.time()
T = {}
for t in TABLES:
    cols = [c for c in MANIFEST["staged_files"][t]["column_order"] if c not in ("profile_text", "profile_embedding")]
    r = gcs("cat", f"{GCS_READ}/{t}.csv.gz", binary=True)
    if r.returncode != 0:
        raise FileNotFoundError(f"{t}.csv.gz: {r.stderr[:200]!r}")
    T[t] = normalize(pd.read_csv(io.BytesIO(gzip.decompress(r.stdout)), header=None, names=cols,
                                 dtype={c: "string" for c in cols if c in TEXT_COLS}, low_memory=False))
    mark = "✅" if len(T[t]) == EXPECTED_ROWS[t] else "🔴"
    print(f"  {mark} {t:<18}{len(T[t]):>10,} rows")
TIMERS["load"] = time.time() - t0
G = T["games"]
UEFA = sorted(c for c in G.competition_id.dropna().unique() if c not in BIG5)
# Display names. competitions.competition_name holds slugs ("uefa-champions-league"), and some
# clubs.club_name values are legal names ("Società Sportiva Lazio S.p.A."); the model copies both.
_COMP_DISPLAY = {"GB1": "Premier League", "ES1": "LaLiga", "IT1": "Serie A", "L1": "Bundesliga",
                 "FR1": "Ligue 1", "CL": "Champions League", "EL": "Europa League"}
_slug = T["competitions"].set_index("competition_id").competition_name.astype(str).to_dict()
COMP_NAME = {c: _COMP_DISPLAY.get(c, " ".join(w.capitalize() for w in str(n).replace("_", "-").split("-")))
             for c, n in _slug.items()}
_tn = pd.concat([T["transfers"][["to_club_id", "to_club_name"]].set_axis(["club_id", "name"], axis=1),
                 T["transfers"][["from_club_id", "from_club_name"]].set_axis(["club_id", "name"], axis=1)]).dropna()
_SHORT = _tn.groupby("club_id").name.agg(lambda x: x.value_counts().index[0]).to_dict()
_LEGAL = re.compile(r"S\.p\.A|Football Club|Kulübü|Club de Fútbol|Futebol Clube|\b(18|19|20)\d\d\b")
CLUB_NAME = {}
for k, v in T["clubs"].set_index("club_id").club_name.items():
    v = str(v)
    short = _SHORT.get(k)
    CLUB_NAME[int(k)] = str(short) if short and (len(v) > 22 or _LEGAL.search(v)) else v
CLUB_DOM = {int(k): str(v) for k, v in T["clubs"].dropna(subset=["domestic_competition_id"])
            .set_index("club_id").domestic_competition_id.items()}
mism = {t: len(T[t]) for t in TABLES if len(T[t]) != EXPECTED_ROWS[t]}
hb(S, "rows: " + ", ".join(f"{t} {len(T[t]):,}" for t in TABLES) +
      ("; all match" if not mism else f"; MISMATCH {mism}"))
hb(S, f"UEFA comps: {UEFA}; load {TIMERS['load']:.0f}s")

---
## 4 · Player-seasons and scouting clubs

The 00 definitions, unchanged: a player-season is one player with appearances in a scope
competition in one season, minutes summed across scope competitions. Only player-seasons with
**900+ minutes** get reports.

For each one, up to **two** scouting clubs are chosen, in this order, always distinct and never
a club he played for that season:

1. **Later signer**—the earliest in-scope club he moved to after his first appearance that
   season, preferring one he didn't also face in Europe.
2. **European opponent**—the club he faced most often in UCL/UEL that season.
3. **Same league**—another club from his league that season (for UEFA-only player-seasons,
   another club with his club's `domestic_competition_id`), picked by a stable hash so reruns
   choose the same club.

In [ ]:
S = "4 · Player-seasons and scouting clubs"
t0 = time.time()
g = G[["game_id", "competition_id", "season", "round_label", "game_date", "home_club_id", "away_club_id",
       "home_club_goals", "away_club_goals", "home_club_position", "away_club_position"]] \
      .rename(columns={"competition_id": "comp"})
A = T["appearances"][["game_id", "player_id", "player_club_id", "minutes_played", "goals", "assists",
                      "yellow_cards", "red_cards"]].merge(g, on="game_id", how="inner")
A["season"] = A["season"].astype(int)
for c in ("minutes_played", "goals", "assists", "yellow_cards", "red_cards"):
    A[c] = A[c].fillna(0).astype(int)
A["is_league"] = A.comp.isin(BIG5)
A["is_euro"] = A.comp.isin(UEFA)
_home = A.player_club_id.eq(A.home_club_id).fillna(False).astype(bool)
A["opp"] = A.away_club_id.where(_home, A.home_club_id)
A["gf"] = A.home_club_goals.where(_home, A.away_club_goals)
A["ga"] = A.away_club_goals.where(_home, A.home_club_goals)
KEY = ["player_id", "season"]

PS = (A.groupby(KEY).agg(apps=("game_id", "size"), minutes=("minutes_played", "sum"),
                          goals=("goals", "sum"), assists=("assists", "sum"),
                          yellow=("yellow_cards", "sum"), red=("red_cards", "sum"),
                          braces=("goals", lambda s: int((s >= 2).sum())),
                          hattricks=("goals", lambda s: int((s >= 3).sum())),
                          league_apps=("is_league", "sum"),
                          first_app=("game_date", "min"), last_app=("game_date", "max"))
        .reset_index())
_lg = (A[A.is_league].groupby(KEY + ["comp"]).size().reset_index(name="n")
       .sort_values(["n", "comp"], ascending=[False, True]).drop_duplicates(KEY))
PS = PS.merge(_lg[KEY + ["comp"]].rename(columns={"comp": "league_comp"}), on=KEY, how="left")
PSD = {(int(r.player_id), int(r.season)): r for r in PS.itertuples(index=False)}   # all player-seasons
P9 = PS[PS.minutes >= MIN_MINUTES].reset_index(drop=True)


def _setagg(df, col):
    d = df.dropna(subset=[col])
    return d.groupby(KEY)[col].agg(lambda s: frozenset(int(x) for x in s.unique())).to_dict()


ps_clubs = _setagg(A, "player_club_id")
eu = A[A.is_euro].dropna(subset=["opp"])
euro_meet = eu.groupby(KEY + ["opp"]).size()
euro_opps = defaultdict(dict)
for (pid, s, opp), n in euro_meet.items():
    euro_opps[(pid, s)][int(opp)] = int(n)

_lgm = g[g.comp.isin(BIG5)]
_lc = pd.concat([_lgm[["comp", "season", "home_club_id"]].rename(columns={"home_club_id": "club"}),
                 _lgm[["comp", "season", "away_club_id"]].rename(columns={"away_club_id": "club"})]).dropna()
_lc["season"] = _lc.season.astype(int)
league_clubs = _lc.groupby(["comp", "season"]).club.agg(lambda s: sorted({int(x) for x in s})).to_dict()
clubs_by_dom = defaultdict(list)
for c, d in CLUB_DOM.items():
    clubs_by_dom[d].append(c)

_tr = T["transfers"][["player_id", "transfer_date", "to_club_id"]].dropna()
_m = P9[KEY + ["first_app"]].merge(_tr, on="player_id")
_m = _m[_m.transfer_date > _m.first_app].sort_values(KEY + ["transfer_date"])
later = defaultdict(list)
for r in _m.itertuples(index=False):
    later[(r.player_id, r.season)].append(int(r.to_club_id))


def stable_pick(options, salt):
    options = sorted(options)
    h = int(hashlib.md5(salt.encode()).hexdigest(), 16)
    return options[h % len(options)]


PCODE = {"later_signer": "ls", "european_opponent": "eo", "same_league": "sl"}
REPORTS = []
avail = Counter()
for r in P9.itertuples(index=False):
    key = (int(r.player_id), int(r.season))
    own = ps_clubs.get(key, frozenset())
    picks, used = [], set()
    L = [c for c in later.get(key, []) if c not in own]
    E = {c: n for c, n in euro_opps.get(key, {}).items() if c not in own}
    if isinstance(r.league_comp, str):
        Sx, s_src = set(league_clubs.get((r.league_comp, r.season), [])) - own, "big5"
    else:
        Sx = set()
        for c in own:
            Sx |= set(clubs_by_dom.get(CLUB_DOM.get(c), []))
        Sx -= own
        s_src = "dom"
    if L:
        pick = next((c for c in L if c not in E), L[0])
        picks.append(("later_signer", pick)); used.add(pick)
    E2 = {c: n for c, n in E.items() if c not in used}
    if E2:
        pick = sorted(E2.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
        picks.append(("european_opponent", pick)); used.add(pick)
    S2 = Sx - used
    if S2:
        picks.append(("same_league", stable_pick(S2, f"{key[0]}-{key[1]}")))
    avail[len(picks)] += 1
    for persp, club in picks[:SCOUTS_PER_PS]:
        REPORTS.append({"report_id": f"{key[0]}-{key[1]}-{PCODE[persp]}",
                        "player_id": key[0], "season": key[1], "perspective": persp,
                        "scout_club_id": int(club), "league_src": s_src})
R = pd.DataFrame(REPORTS)
TIMERS["perspectives"] = time.time() - t0

print(f"player-seasons >= {MIN_MINUTES} min: {len(P9):,}")
print(f"reports: {len(R):,}   (00 measured {EXPECTED_REPORTS:,}"
      f"{'' if len(R) == EXPECTED_REPORTS else '—DIFFERENT, see handback'})")
print("perspectives available per player-season: " + "  ".join(f"{k}:{v:,}" for k, v in sorted(avail.items())))
print("\nreports by perspective:")
for k, v in R.perspective.value_counts().items():
    print(f"   {k:<20}{v:>8,}")
by_season = R.groupby("season").size()
print("\nreports by season: " + "  ".join(f"{s}:{n:,}" for s, n in by_season.items()))
hb(S, f"ps>={MIN_MINUTES}: {len(P9):,}; reports {len(R):,} (expected {EXPECTED_REPORTS:,}); "
      + "; ".join(f"{k} {v:,}" for k, v in R.perspective.value_counts().items()))
hb(S, "by season: " + "  ".join(f"{s}:{n:,}" for s, n in by_season.items()))

---
## 5 · Fact blocks

Every report is generated from a fact block built here, and nothing else. All facts are **as of
30 June** at the end of the season: valuations and transfers after that date are never used.

What a fact block carries:

- **Scouting club** and its league position that season, plus the perspective it writes from.
- **Player**: name, age, citizenship, position, foot, and height only when it is
  notable (190 cm or more, 170 cm or less).
- **Season**: clubs played for (with dates for a mid-season move), totals and per-competition
  lines in covered matches, goals per 90, multi-goal matches, cards.
- **Club context**: final league position, the last European round reached, and (v1.3.0) the
  coach or coaches and home ground that season, from `games`, for both the player's club and the
  scouting club. Birthplace is added to the player lines. These are the "well-known" club facts a
  scout would use, taken from the data as of the season so the model doesn't have to recall them.
- **Absences** (00's primary definition) and a season-ending absence, cause not recorded.
- **Market value** at the start and end of the season, and **how he arrived** at his club.
- **Previous season** in covered matches, and **matches against the scouting club** this season.
- **Match detail** (v1.2.0), from `game_events`: starts versus substitute appearances, goals that
  opened the scoring, put his side ahead or won the match, late goals, how he scored (penalty,
  header, foot), home and away, his most frequent assist partners, and his main club's results
  with and without him.

The event data's conventions are **detected, not assumed**: which id in a substitution is the
player coming on, how own goals are attributed (whichever matches more final scores), and whether
goal descriptions carry a usable finish type. Lead and match-winner facts come only from matches
whose goal events add up to the final score. Every detection is printed and sent in the handback.

Coverage is stated in every block: Big 5 leagues, Champions League and Europa League only.

In [ ]:
S = "5 · Fact blocks"
t0 = time.time()
PL = T["players"].set_index("player_id")
PLD = {int(r.player_id): r for r in T["players"].itertuples(index=False)}

# per player-season, per competition
_pc = A.groupby(KEY + ["comp"]).agg(apps=("game_id", "size"), minutes=("minutes_played", "sum"),
                                    goals=("goals", "sum"), assists=("assists", "sum")).reset_index()
PSC = defaultdict(list)
for r in _pc.sort_values(["apps"], ascending=False).itertuples(index=False):
    PSC[(int(r.player_id), int(r.season))].append((r.comp, int(r.apps), int(r.minutes), int(r.goals), int(r.assists)))

# clubs per player-season, in date order
_pcl = A.dropna(subset=["player_club_id"]).groupby(KEY + ["player_club_id"]).agg(
    apps=("game_id", "size"), first=("game_date", "min"), last=("game_date", "max")).reset_index()
PCL = defaultdict(list)
for r in _pcl.sort_values("first").itertuples(index=False):
    PCL[(int(r.player_id), int(r.season))].append((int(r.player_club_id), int(r.apps), r.first, r.last))

# club final league position and European run
_cg = pd.concat([
    g.assign(club=g.home_club_id, pos=g.home_club_position)[["club", "season", "comp", "game_date", "round_label", "pos"]],
    g.assign(club=g.away_club_id, pos=g.away_club_position)[["club", "season", "comp", "game_date", "round_label", "pos"]],
]).dropna(subset=["club"])
_cg["season"] = _cg.season.astype(int)
_cg = _cg.sort_values("game_date")
_lastl = _cg[_cg.comp.isin(BIG5)].groupby(["club", "season"]).last()
CLUB_POS = {(int(c), int(s)): (r.comp, (None if pd.isna(r.pos) else int(r.pos))) for (c, s), r in _lastl.iterrows()}
_laste = _cg[_cg.comp.isin(UEFA)].groupby(["club", "season", "comp"]).last()
EURO_RUN = defaultdict(list)
for (c, s, comp), r in _laste.iterrows():
    EURO_RUN[(int(c), int(s))].append((comp, str(r.round_label)))

# absences (00's definition)
cg = _cg[["club", "season", "game_date"]].copy()
cgid = pd.concat([g[["game_id", "season", "game_date", "home_club_id"]].rename(columns={"home_club_id": "club"}),
                  g[["game_id", "season", "game_date", "away_club_id"]].rename(columns={"away_club_id": "club"})]).dropna(subset=["club"])
cgid["season"] = cgid.season.astype(int)
cgid = cgid.sort_values(["club", "season", "game_date", "game_id"])
cgid["idx"] = cgid.groupby(["club", "season"]).cumcount()
club_last_idx = cgid.groupby(["club", "season"]).idx.max()
SEG = ["player_id", "season", "player_club_id"]
a = A[["player_id", "season", "player_club_id", "game_id", "game_date"]].dropna(subset=["player_club_id"]) \
      .merge(cgid[["game_id", "club", "idx"]], left_on=["game_id", "player_club_id"], right_on=["game_id", "club"])
a = a.sort_values(SEG + ["idx"])
grp = a.groupby(SEG)
a["seg_apps"] = grp.idx.transform("size")
a["gap"] = grp.idx.diff() - 1
a["gap_days"] = grp.game_date.diff().dt.days
a["gap_from"] = grp.game_date.shift()
_gp, _gd = PRIMARY_ABSENCE
gaps = a[(a.seg_apps >= ABSENCE_MIN_APPS) & (a.gap >= _gp) & (a.gap_days >= _gd)]
ABS = defaultdict(list)
for r in gaps.itertuples(index=False):
    ABS[(int(r.player_id), int(r.season))].append((int(r.player_club_id), r.gap_from, r.game_date, int(r.gap), int(r.gap_days)))
seg_end = a.groupby(SEG).agg(last_idx=("idx", "max"), last_date=("game_date", "max"), n=("idx", "size")).reset_index()
_psl = a.groupby(KEY).game_date.max().rename("ps_last")
seg_end = seg_end.merge(_psl, on=KEY)
seg_end = seg_end[(seg_end.last_date == seg_end.ps_last) & (seg_end.n >= ABSENCE_MIN_APPS)]
TRAIL = {}
for r in seg_end.itertuples(index=False):
    cl = club_last_idx.get((r.player_club_id, r.season))
    if cl is not None and cl - r.last_idx >= TRAILING_ABSENCE_MIN:
        TRAIL[(int(r.player_id), int(r.season))] = (int(r.player_club_id), r.last_date, int(cl - r.last_idx))

# market value at season start / end (as-of, never after 30 June)
pv = T["player_valuations"][["player_id", "valuation_date", "market_value_in_eur"]].dropna().sort_values("valuation_date")
pv["player_id"] = pv.player_id.astype("int64")
q = P9[KEY].copy()
q["player_id"] = q.player_id.astype("int64")
q["d_start"] = pd.to_datetime(q.season.astype(str) + "-08-31").astype("datetime64[ns]")
q["d_end"] = pd.to_datetime((q.season + 1).astype(str) + "-06-30").astype("datetime64[ns]")
pv["valuation_date"] = pv.valuation_date.astype("datetime64[ns]")
VAL = {}
_vs = pd.merge_asof(q.sort_values("d_start"), pv.rename(columns={"valuation_date": "d_start"}), on="d_start",
                    by="player_id", direction="backward")
_ve = pd.merge_asof(q.sort_values("d_end"), pv.rename(columns={"valuation_date": "d_end"}), on="d_end",
                    by="player_id", direction="backward")
_vstart = {(int(r.player_id), int(r.season)): r.market_value_in_eur for r in _vs.itertuples(index=False)}
for r in _ve.itertuples(index=False):
    VAL[(int(r.player_id), int(r.season))] = (_vstart.get((int(r.player_id), int(r.season))), r.market_value_in_eur)

# How he arrived: the latest move INTO a club he played for this season, dated on or before
# 30 June. (v1.0 took the latest move of any kind, which let a 30 June loan return read as a
# completed signing: "having now completed his move to Atalanta".)
_own = pd.DataFrame([(p_, s_, c_) for (p_, s_), cs in ps_clubs.items() for c_ in cs],
                    columns=["player_id", "season", "to_club_id"]).astype("int64")
_own = _own.merge(q[["player_id", "season", "d_end"]].astype({"season": "int64"}), on=["player_id", "season"])
trx = T["transfers"][["player_id", "transfer_date", "from_club_id", "from_club_name", "to_club_id",
                      "to_club_name", "transfer_fee"]].dropna(subset=["transfer_date", "to_club_id"]).copy()
trx = trx.astype({"player_id": "int64", "to_club_id": "int64"})
trx["transfer_date"] = trx.transfer_date.astype("datetime64[ns]")
_tri = _own.merge(trx, on=["player_id", "to_club_id"])
_tri = _tri[_tri.transfer_date <= _tri.d_end].sort_values("transfer_date").drop_duplicates(["player_id", "season"], keep="last")
TRIN = {}
for r in _tri.itertuples(index=False):
    _from = CLUB_NAME.get(int(r.from_club_id), str(r.from_club_name)) if pd.notna(r.from_club_id) else str(r.from_club_name)
    TRIN[(int(r.player_id), int(r.season))] = (_from, CLUB_NAME.get(int(r.to_club_id), str(r.to_club_name)),
                                               r.transfer_date, r.transfer_fee)

# matches against each opponent this season
_need = R[["player_id", "season", "scout_club_id"]].rename(columns={"scout_club_id": "opp"}).drop_duplicates()
_h = A.dropna(subset=["opp"]).copy()
_h["player_id"] = _h.player_id.astype("int64"); _h["opp"] = _h.opp.astype("int64")
_h = _h.merge(_need.astype("int64"), on=["player_id", "season", "opp"])
_h["at_home"] = _h.player_club_id.eq(_h.home_club_id).fillna(False).astype(int)
_h = _h.merge(G[["game_id", "stadium"]], on="game_id", how="left").sort_values("game_date", kind="mergesort")
_h["venue"] = [(f"{st} (his side at home)" if hm else f"{st} (away)") if pd.notna(st) and str(st).strip()
               else ("his side at home" if hm else "away") for st, hm in zip(_h.stadium, _h.at_home)]
for _k, _op in (("W", _h.gf > _h.ga), ("D", _h.gf == _h.ga), ("L", _h.gf < _h.ga)):
    _h[_k] = _op.fillna(False).astype(int)        # a match with no recorded score counts as none of W/D/L
_hh = _h.groupby(KEY + ["opp"]).agg(n=("game_id", "size"), m=("minutes_played", "sum"), gl=("goals", "sum"),
                                    a=("assists", "sum"), W=("W", "sum"), D=("D", "sum"), L=("L", "sum"),
                                    comps=("comp", lambda x: tuple(sorted(set(x)))),
                                    hm=("at_home", "sum"), venues=("venue", tuple)).reset_index()
H2H = {(int(r.player_id), int(r.season), int(r.opp)): (int(r.n), int(r.m), int(r.gl), int(r.a), int(r.W), int(r.D),
                                                        int(r.L), r.comps, int(r.hm), r.venues) for r in _hh.itertuples(index=False)}

# ---- Match-event detail (v1.2.0) --------------------------------------------------------
# Every fact below is derived from game_events/appearances/games and checked against the data
# before it is trusted. Conventions the raw tables don't document are DETECTED, not assumed,
# and each detection is reported in the handback.
EVT = {}                 # what we learned about the event data, for the handback
EV = T["game_events"].merge(g[["game_id", "season", "home_club_id", "away_club_id",
                               "home_club_goals", "away_club_goals"]], on="game_id", how="inner")
EV["season"] = EV.season.astype(int)
EV["minute"] = pd.to_numeric(EV.minute, errors="coerce")
EV["desc"] = EV.description.fillna("").astype(str)
EV["etype"] = EV.event_type.fillna("").astype(str).str.lower()

# -- substitutions: which id is the player coming ON? The one with the lower median minutes.
_sub = EV[EV.etype.str.startswith("substitution")]
_apm = A[["game_id", "player_id", "minutes_played"]].dropna().astype({"player_id": "int64"})


def _median_minutes(col):
    k = _sub[["game_id", col]].dropna().rename(columns={col: "player_id"}).astype({"player_id": "int64"})
    m = k.merge(_apm, on=["game_id", "player_id"])
    return (float(m.minutes_played.median()) if len(m) else float("nan")), k


_m_in, _k_in = _median_minutes("player_in_id")
_m_pid, _k_pid = _median_minutes("player_id")
if np.isnan(_m_in) and np.isnan(_m_pid):
    CAME_ON, WENT_OFF = set(), set()
    EVT["substitutions"] = "no usable substitution events"
else:
    on_k, off_k = ((_k_in, _k_pid) if (np.nan_to_num(_m_in, nan=999) <= np.nan_to_num(_m_pid, nan=999))
                   else (_k_pid, _k_in))
    CAME_ON = set(zip(on_k.game_id, on_k.player_id))
    WENT_OFF = set(zip(off_k.game_id, off_k.player_id))
    EVT["substitutions"] = (f"player_in_id median {_m_in:.0f} min, player_id median {_m_pid:.0f} min -> "
                            f"'came on' = {'player_in_id' if on_k is _k_in else 'player_id'}")
_ak = A[["game_id", "player_id", "season", "player_club_id", "minutes_played", "goals", "home_club_id"]].copy()
_ak["player_id"] = _ak.player_id.astype("int64")
_ak["on"] = [(gm, p) in CAME_ON for gm, p in zip(_ak.game_id, _ak.player_id)]
_ak["off"] = [(gm, p) in WENT_OFF for gm, p in zip(_ak.game_id, _ak.player_id)]
_ak["home"] = _ak.player_club_id.eq(_ak.home_club_id).fillna(False).astype(bool)
_ak["h_goal"] = _ak.goals.fillna(0).astype(int) * _ak.home.astype(int)
_sa = _ak.groupby(KEY).agg(on=("on", "sum"), off=("off", "sum"), h_apps=("home", "sum"), h_goals=("h_goal", "sum"),
                           goals=("goals", "sum"), apps=("game_id", "size"))
SUBS = {(int(p), int(s_)): (int(r.on), int(r.off)) for (p, s_), r in _sa.iterrows()}
HOMEAWAY = {(int(p), int(s_)): (int(r.h_apps), int(r.h_goals), int(r.apps - r.h_apps), int(r.goals - r.h_goals))
            for (p, s_), r in _sa.iterrows()}

# -- goals: own goals, and the attribution convention that makes tallies match final scores
GE = EV[EV.etype.str.startswith("goal")].copy()
GE["own"] = GE.desc.str.contains(r"own[\s-]?goal", case=False, regex=True)
GE = GE.sort_values(["game_id", "minute", "game_event_id"], kind="mergesort")
_hc, _ac = GE.home_club_id.astype("Int64"), GE.away_club_id.astype("Int64")
_other = _ac.where(GE.club_id.eq(_hc).fillna(False), _hc)


def _consistent(credit):
    t = GE.assign(credit=credit)
    tal = t.groupby(["game_id", "credit"]).size().unstack(fill_value=0)
    gg = g.set_index("game_id")[["home_club_id", "away_club_id", "home_club_goals", "away_club_goals"]].dropna()
    ok = set()
    for gid, r in gg.iterrows():
        if gid not in tal.index:
            if r.home_club_goals == 0 and r.away_club_goals == 0:
                ok.add(gid)
            continue
        row = tal.loc[gid]
        if (int(row.get(r.home_club_id, 0)) == int(r.home_club_goals)
                and int(row.get(r.away_club_id, 0)) == int(r.away_club_goals)):
            ok.add(gid)
    return ok


_cred_a = GE.club_id.astype("Int64")                       # own goal credited to club_id
_cred_b = _cred_a.where(~GE.own, _other)                   # own goal credited to the other side
_ok_a, _ok_b = _consistent(_cred_a), _consistent(_cred_b)
GE["credit"], CONSISTENT = (_cred_b, _ok_b) if len(_ok_b) >= len(_ok_a) else (_cred_a, _ok_a)
EVT["goals"] = (f"{len(GE):,} goal events, {int(GE.own.sum()):,} own goals; tallies match the final score in "
                f"{len(CONSISTENT):,} of {G.game_id.nunique():,} games "
                f"(own goal credited to {'the other side' if len(_ok_b) >= len(_ok_a) else 'club_id'})")

opener, ahead, winner = [], [], []
for gid, d in GE.groupby("game_id", sort=False):
    tally = Counter()
    hcl, acl = d.home_club_id.iloc[0], d.away_club_id.iloc[0]
    hf, af = d.home_club_goals.iloc[0], d.away_club_goals.iloc[0]
    first = True
    for r in d.itertuples(index=False):
        c = r.credit
        if pd.isna(c) or pd.isna(hcl) or pd.isna(acl):
            opener.append(False); ahead.append(False); winner.append(False)
            first = False
            continue
        c, hcl, acl = int(c), int(hcl), int(acl)
        opp = acl if c == hcl else hcl
        before, before_opp = tally[c], tally[opp]
        tally[c] += 1
        own = bool(r.own)
        opener.append(first and not own)
        first = False
        if gid in CONSISTENT and not own:
            final_c = hf if c == hcl else af
            final_o = af if c == hcl else hf
            ahead.append(tally[c] > tally[opp] and before <= before_opp)
            winner.append(final_c > final_o and tally[c] == final_o + 1 and before == final_o)
        else:
            ahead.append(False)
            winner.append(False)
GE["opener"], GE["ahead"], GE["winner"] = opener, ahead, winner
GE["late"] = GE.minute >= 76
mine = GE[~GE.own & GE.player_id.notna()].copy()
mine["player_id"] = mine.player_id.astype("int64")
_gd = mine.groupby(KEY).agg(n=("game_id", "size"), opener=("opener", "sum"), ahead=("ahead", "sum"),
                            winner=("winner", "sum"), late=("late", "sum"))
GOALDETAIL = {(int(p), int(s_)): tuple(int(v) for v in r) for (p, s_), r in _gd.iterrows()}

# -- finish types: only if most goal descriptions carry a recognisable finish
_FIN = {"penalty": r"penalt", "header": r"\bhead", "left foot": r"left[\s-]?foot",
        "right foot": r"right[\s-]?foot", "direct free kick": r"free[\s-]?kick"}
# "Right-footed shot, 1. Goal of the Season Assist: Cross" -> keep only the text before "Assist",
# so the assist type (a headed pass, a cross) is never counted as his finish.
mine["fdesc"] = mine.desc.str.replace(r"(?is)\bassist\b.*$", "", regex=True)
_known = mine.fdesc.str.contains("|".join(_FIN.values()), case=False, regex=True)
FIN_COVERAGE = float(_known.mean()) if len(mine) else 0.0
FIN_OK = FIN_COVERAGE >= 0.5
FINISH = {}
if FIN_OK:
    for k, pat in _FIN.items():
        mine[k] = mine.fdesc.str.contains(pat, case=False, regex=True)
    _fd = mine[_known].groupby(KEY)[list(_FIN)].sum()
    FINISH = {(int(p), int(s_)): {k: int(v) for k, v in r.items() if v} for (p, s_), r in _fd.iterrows()}
_frag = Counter(x.strip() for dsc in mine.fdesc.head(20000) for x in dsc.split(",") if x.strip())
EVT["finishes"] = (f"descriptions with a recognisable finish: {FIN_COVERAGE:.0%} -> finish line "
                   f"{'ON' if FIN_OK else 'OFF'}; top fragments: " +
                   "; ".join(f"{k} ({v})" for k, v in _frag.most_common(15)))

# -- assist partnerships (in-scope players only, at least 2 combinations)
_pa = mine.dropna(subset=["player_assist_id"]).copy()
_pa["assister"] = _pa.player_assist_id.astype("int64")
_pa = _pa[_pa.assister.isin(PLD.keys()) & _pa.assister.ne(_pa.player_id)]   # v1.2 paired Mbappé with himself
SET_UP_BY, SET_UP = {}, {}
for (p, s_, a_), n in _pa.groupby(["player_id", "season", "assister"]).size().items():
    if n >= 2 and n > SET_UP_BY.get((int(p), int(s_)), (None, 0))[1]:
        SET_UP_BY[(int(p), int(s_))] = (int(a_), int(n))
    if n >= 2 and n > SET_UP.get((int(a_), int(s_)), (None, 0))[1]:
        SET_UP[(int(a_), int(s_))] = (int(p), int(n))

# -- the main club's results with and without him (covered matches, at least 3 without)
_cr = pd.concat([
    g.assign(club=g.home_club_id, gf=g.home_club_goals, ga=g.away_club_goals),
    g.assign(club=g.away_club_id, gf=g.away_club_goals, ga=g.home_club_goals),
])[["club", "season", "game_id", "gf", "ga"]].dropna()
_cr = _cr.astype({"club": "int64", "season": "int64", "gf": "int64", "ga": "int64"})
CLUB_GAMES = {k: list(zip(d.game_id, d.gf, d.ga)) for k, d in _cr.groupby(["club", "season"])}
_played = A.dropna(subset=["player_club_id"]).groupby(KEY + ["player_club_id"]).game_id.agg(set).to_dict()


def with_without(pid, s, club):
    games = CLUB_GAMES.get((club, s))
    mineg = _played.get((pid, s, club))
    if not games or not mineg:
        return None
    w = [(gf, ga) for gm, gf, ga in games if gm in mineg]
    o = [(gf, ga) for gm, gf, ga in games if gm not in mineg]
    if len(o) < 3 or not w:
        return None

    def rec(rows):
        return (sum(gf > ga for gf, ga in rows), sum(gf == ga for gf, ga in rows), sum(gf < ga for gf, ga in rows),
                sum(gf for gf, _ in rows) / len(rows), sum(ga for _, ga in rows) / len(rows), len(rows))
    return rec(w), rec(o)


# -- coaches and home grounds as of the season (from games), so the model never has to recall them
_mg = pd.concat([G[["season", "game_date", "home_club_id", "home_club_manager_name"]].set_axis(["season", "d", "club", "mgr"], axis=1),
                 G[["season", "game_date", "away_club_id", "away_club_manager_name"]].set_axis(["season", "d", "club", "mgr"], axis=1)])
_mg = _mg.dropna(subset=["season", "club", "mgr"])
_mg = _mg[_mg.mgr.astype(str).str.strip().ne("")].astype({"club": "int64", "season": "int64"})
COACH = {}
for (c_, s_), d in _mg.sort_values("d", kind="mergesort").groupby(["club", "season"]):
    cnt = d.mgr.value_counts()
    order = [str(m) for m in d.drop_duplicates("mgr").mgr if cnt[m] >= 3]
    if order:
        COACH[(int(c_), int(s_))] = order
_st = G[["season", "home_club_id", "stadium"]].dropna()
_st = _st[_st.stadium.astype(str).str.strip().ne("")].astype({"home_club_id": "int64", "season": "int64"})
GROUND = {(int(c_), int(s_)): str(v) for (c_, s_), v in
          _st.groupby(["home_club_id", "season"]).stadium.agg(lambda x: x.value_counts().index[0]).items()}


def club_extra(club, s):
    bits = []
    co = COACH.get((club, s))
    if co:
        bits.append(f"coach {co[0]}" if len(co) == 1 else "coaches " + ", then ".join(co))
    if GROUND.get((club, s)):
        bits.append(f"home ground {GROUND[(club, s)]}")
    return "; ".join(bits)


_pl = T["players"]
_bp = (_pl.city_of_birth.notna().mean() if "city_of_birth" in _pl else 0.0)
EVT["club and player context"] = (f"coach named on {G.home_club_manager_name.notna().mean():.0%} of games, "
                                  f"stadium on {G.stadium.notna().mean():.0%}; {len(COACH):,} club-seasons with a coach, "
                                  f"{len(GROUND):,} with a home ground; birthplace for {_bp:.0%} of players")

print("event checks:")
for k, v in EVT.items():
    print(f"   {k}: {v}")


# future clubs (for leakage QA only—never shown to the model)
_fut = T["transfers"][["player_id", "transfer_date", "to_club_name"]].dropna()


def pl(n, word):
    return f"{n} {word}" + ("" if n == 1 else "s")


def fdate(d):
    return pd.Timestamp(d).strftime("%-d %b %Y") if pd.notna(d) else "?"


def league_phrase(club, s):
    lp = CLUB_POS.get((club, s))
    if not lp:
        dom = CLUB_DOM.get(club)
        if dom in BIG5:          # a Big 5 club with no league matches that season was not in that league
            return f"outside {COMP_NAME.get(dom, dom)} this season"
        return f"plays in {COMP_NAME.get(dom, dom)} (league matches not covered)" if dom else "league not covered"
    comp, pos = lp
    return f"{COMP_NAME.get(comp, comp)}, finished {pos}{'st' if pos == 1 else 'nd' if pos == 2 else 'rd' if pos == 3 else 'th'}" if pos else f"{COMP_NAME.get(comp, comp)}"


def round_phrase(label):
    l = re.sub(r"\s*(1st|2nd)\s+leg", "", str(label), flags=re.I).strip().lower()
    for key, out in (("group", "group stage"), ("league phase", "league phase"), ("league stage", "league phase"),
                     ("semi", "semi-finals"), ("quarter", "quarter-finals"), ("last 16", "round of 16"),
                     ("round of 16", "round of 16"), ("last 32", "knockout play-offs"),
                     ("knockout", "knockout play-offs"), ("intermediate", "knockout play-offs"),
                     ("qualif", "qualifying rounds"), ("play-off", "play-off round")):
        if key in l:
            return out
    return "final" if l == "final" else l


def facts_for(rep):
    pid, s, scout = rep["player_id"], rep["season"], rep["scout_club_id"]
    key = (pid, s)
    p = PLD[pid]
    ps = PSD[key]
    start = pd.Timestamp(f"{s}-09-01")
    dob = p.date_of_birth
    age = (start.year - dob.year - ((start.month, start.day) < (dob.month, dob.day))) if pd.notna(dob) else None
    end = pd.Timestamp(f"{s + 1}-06-30")
    age_end = (end.year - dob.year - ((end.month, end.day) < (dob.month, dob.day))) if pd.notna(dob) else None
    L = []
    _sx = club_extra(scout, s)
    L.append(f"SCOUTING CLUB: {CLUB_NAME.get(scout, scout)} ({league_phrase(scout, s)}{'; ' + _sx if _sx else ''})")
    L.append(f"SEASON: {season_label(s)}. Facts are as of 30 June {s + 1}. Covered matches: Big 5 leagues, "
             f"UEFA Champions League and UEFA Europa League only; domestic cups and other leagues are not included.")
    L.append(f"PLAYER: {p.player_name}")
    if age_end is not None:      # a plain age: any date beside it ("27 on 30 June, the date of this
        L.append(f"Age: {age_end}")  # report") reads as a birthday and comes back as "turns 27 today"
    _city, _ctry = getattr(p, "city_of_birth", None), getattr(p, "country_of_birth", None)
    _born = (", ".join(str(x) for x in (_city, _ctry) if pd.notna(x) and str(x).strip())
             if pd.notna(_city) and str(_city).strip() else "")      # a birthplace needs a city
    L.append(f"Citizenship: {p.country_of_citizenship if pd.notna(p.country_of_citizenship) else 'unknown'}" +
             (f"; born in {_born}" if _born else ""))
    pos = ", ".join(str(x) for x in (p.detailed_position, p.main_position) if pd.notna(x))
    extra = []
    if pd.notna(p.foot):
        extra.append(f"foot: {p.foot}")
    if pd.notna(p.height_in_cm) and not 170 < int(p.height_in_cm) < 190:   # notable heights only
        extra.append(f"height: {int(p.height_in_cm)} cm")
    L.append(f"Position: {pos or 'unknown'}" + (f"; {'; '.join(extra)}" if extra else ""))
    cl = PCL.get(key, [])
    if len(cl) > 1:
        L.append("Clubs this season: " + "; ".join(
            f"{CLUB_NAME.get(c, c)} ({n} apps, {pd.Timestamp(f0).strftime('%b %Y')}–{pd.Timestamp(f1).strftime('%b %Y')})"
            for c, n, f0, f1 in cl))
    elif cl:
        L.append(f"Club this season: {CLUB_NAME.get(cl[0][0], cl[0][0])}")
    p90 = ps.goals * 90 / ps.minutes if ps.minutes else 0
    L.append(f"Season totals (covered matches): {pl(ps.apps, 'appearance')}, {ps.minutes:,} minutes, {pl(ps.goals, 'goal')}, "
             f"{pl(ps.assists, 'assist')}, {pl(ps.yellow, 'yellow card')}, {pl(ps.red, 'red card')}; "
             f"{p90:.2f} goals per 90 minutes")
    L.append("By competition: " + "; ".join(f"{COMP_NAME.get(c, c)}: {n} apps, {m:,} min, {pl(gl, 'goal')}, {pl(asst, 'assist')}"
                                            for c, n, m, gl, asst in PSC.get(key, [])))
    if ps.braces:
        L.append(f"Matches with 2+ goals: {ps.braces}" + (f" (including {ps.hattricks} with 3+)" if ps.hattricks else ""))
    on, off = SUBS.get(key, (0, 0))
    L.append(f"Starts: {ps.apps - on} of {ps.apps} appearances; came on as a substitute {pl(on, 'time')}; "
             f"substituted off {pl(off, 'time')}")
    gd = GOALDETAIL.get(key)
    if gd and gd[0]:
        n, op, ah, wn, lt = gd
        parts = [(ah, "gave his side the lead", "gave his side the lead"),
                 (op, "was the first goal of the match", "were the first goal of the match"),
                 (wn, "was the match-winner (the goal that decided the result)",
                  "were match-winners (the goal that decided the result)"),
                 (lt, "came in the last 15 minutes", "came in the last 15 minutes")]
        parts = [f"{v} {one if v == 1 else many}" for v, one, many in parts if v]
        if parts:
            L.append(f"His {pl(n, 'goal')} in detail (one goal can count under more than one heading; "
                     f"do not add these together): " + "; ".join(parts))
    fin = FINISH.get(key)
    if fin:
        _lab = {"penalty": ("penalty", "penalties"), "header": ("header", "headers"),
                "left foot": ("with his left foot",) * 2, "right foot": ("with his right foot",) * 2,
                "direct free kick": ("direct free kick", "direct free kicks")}
        L.append("How he scored (one goal can count twice, e.g. a right-footed penalty): " +
                 ", ".join(f"{v} {_lab[k][0 if v == 1 else 1]}" for k, v in fin.items()))
    ha = HOMEAWAY.get(key)
    if ha:
        L.append(f"Home and away: {pl(ha[0], 'home appearance')} ({pl(ha[1], 'goal')}), "
                 f"{pl(ha[2], 'away appearance')} ({pl(ha[3], 'goal')})")
    sb = SET_UP_BY.get(key)
    if sb:
        L.append(f"Set up most often by: {PLD[sb[0]].player_name} ({sb[1]} goals)")
    su = SET_UP.get(key)
    if su:
        L.append(f"Teammate he set up most: {PLD[su[0]].player_name} ({su[1]} goals)")
    if cl:
        main = max(cl, key=lambda x: x[1])[0]
        ww = with_without(pid, s, main)
        if ww:
            (w1, d1, l1, f1, a1, n1), (w2, d2, l2, f2, a2, n2) = ww
            L.append(f"{CLUB_NAME.get(main, main)} with him: won {w1}, drew {d1}, lost {l1}, {f1:.1f} goals for and "
                     f"{a1:.1f} against per match; without him ({pl(n2, 'match').replace('matchs', 'matches')}): "
                     f"won {w2}, drew {d2}, lost {l2}, {f2:.1f} for and {a2:.1f} against")
    ctx = []
    for c, *_ in cl:
        bits = [league_phrase(c, s)] + ([club_extra(c, s)] if club_extra(c, s) else [])
        for comp, rnd in EURO_RUN.get((c, s), []):
            bits.append(f"{COMP_NAME.get(comp, comp)}: furthest round reached, {round_phrase(rnd)}")
        ctx.append(f"{CLUB_NAME.get(c, c)}—" + "; ".join(bits))
    if ctx:
        L.append("Club context: " + " | ".join(ctx))
    ab = ABS.get(key, [])
    tr = TRAIL.get(key)
    if ab or tr:
        parts = [f"missed {n} consecutive {CLUB_NAME.get(c, c)} matches between {fdate(d0)} and {fdate(d1)} ({dd} days)"
                 for c, d0, d1, n, dd in ab]
        if tr:
            parts.append(f"did not appear in {CLUB_NAME.get(tr[0], tr[0])}'s final {tr[2]} covered matches after {fdate(tr[1])}")
        L.append("Absences from the side (cause not recorded): " + "; ".join(parts))
    v0, v1 = VAL.get(key, (None, None))
    if pd.notna(v1) or pd.notna(v0):
        L.append(f"Market value: {money(v0)} at the start of the season, {money(v1)} at the end")
    t = TRIN.get(key)
    if t:
        L.append(f"Most recent move: from {t[0]} to {t[1]} on {fdate(t[2])} (fee: {money(t[3])})")
    if (pid, s - 1) in PSD:
        pp = PSD[(pid, s - 1)]
        L.append(f"Previous season ({season_label(s - 1)}, covered matches): {pl(pp.apps, 'appearance')}, "
                 f"{pp.minutes:,} minutes, {pl(pp.goals, 'goal')}, {pl(pp.assists, 'assist')}")
    else:
        L.append(f"Previous season ({season_label(s - 1)}): no covered matches")
    h = H2H.get((pid, s, scout))
    if h:
        L.append(f"Against {CLUB_NAME.get(scout, scout)} this season: {pl(h[0], 'match').replace('matchs', 'matches')} "
                 f"({', '.join(COMP_NAME.get(c, c) for c in h[7])}), {pl(h[1], 'minute')}, {pl(h[2], 'goal')}, "
                 f"{pl(h[3], 'assist')}; his side won {h[4]}, drew {h[5]}, lost {h[6]}; "
                 f"played at: {', '.join(h[9])}")
    else:
        L.append(f"Against {CLUB_NAME.get(scout, scout)} this season: no covered meetings")
    return "\n".join(L)


R = R.merge(P9[KEY + ["minutes"]], on=KEY, how="left")
R["player_name"] = R.player_id.map(PL.player_name).astype(str)
R["player_club_id"] = [max(PCL.get((p, s), [(None, 0, None, None)]), key=lambda x: x[1])[0]
                       for p, s in zip(R.player_id, R.season)]
R["report_date"] = pd.to_datetime((R.season + 1).astype(str) + "-06-30").dt.date
R["facts"] = [facts_for(rep) for rep in R.to_dict("records")]
TIMERS["facts"] = time.time() - t0
_fl = R.facts.str.len()
print(f"{len(R):,} fact blocks in {TIMERS['facts']:.0f}s; length median {_fl.median():,.0f} chars, max {_fl.max():,}")
print("\nExample (the most valuable player's newest report):\n")
_ex = R.assign(v=R.player_id.map(PL.highest_market_value_in_eur)).sort_values(["v", "season"], ascending=False).iloc[0]
print(_ex.facts)
hb(S, f"fact blocks {len(R):,}; median {_fl.median():,.0f} chars; max {_fl.max():,}; {TIMERS['facts']:.0f}s")
for _k, _v in EVT.items():
    hb(S, f"event check, {_k}: {_v}")
_cov = {"starts line": 1.0, "goal detail": sum(1 for k in zip(R.player_id, R.season) if GOALDETAIL.get(k, (0,))[0]) / len(R),
        "finish line": sum(1 for k in zip(R.player_id, R.season) if FINISH.get(k)) / len(R),
        "assist partner": sum(1 for k in zip(R.player_id, R.season) if SET_UP_BY.get(k) or SET_UP.get(k)) / len(R),
        "with/without": float(R.facts.str.contains(" with him: ").mean())}
hb(S, "share of fact blocks with each detail line: " + ", ".join(f"{k} {v:.0%}" for k, v in _cov.items()))

---
## 6 · The prompt

One system instruction for every report, and a short perspective line that changes with the
scouting club's relationship to the player. The response is structured JSON with three fields:
`headline`, `report_text`, `recommendation` (SIGN, MONITOR or PASS).

Rule 8 keeps the prompt's own plumbing out of the prose: a scout never says "in the covered
matches". Section 7 checks for exactly that.

In [ ]:
SYSTEM = (
    "You are a professional football scout writing an internal scouting report for your club's "
    "sporting director. Write in English, in the plain, specific voice of a working scout.\n\n"
    "Rules:\n"
    "1. Everything about the player comes only from the facts: his matches, numbers, moves, fitness, "
    "international career and personal life. Do not add matches, opponents, injuries, quotes, "
    "transfer rumours, wages, contracts, nicknames or anything else about him that is not in the "
    "facts. About either club you may use well-known facts that do not change from season to season "
    "(its city or region, its rivals, its identity or the way it recruits), but nothing time-bound "
    "that is not in the facts: no other coaches, owners, finances, trophies or other players.\n"
    "2. You are writing at the end of the season named in the facts. You know nothing that happened "
    "later: never mention later seasons, later transfers, or later fees.\n"
    "3. Absences from the side have no recorded cause. Describe them only as absences (for example "
    "'missed nine matches between November and January'). Never call them injuries, suspensions, "
    "loss of form, or being dropped or unused.\n"
    "4. The first sentence names the player exactly as written in the facts, with his club and his "
    "age, so he cannot be confused with another player of the same name. Give the age plainly "
    "('the 27-year-old', 'aged 27'); never say when he turned or will turn that age. "
    "Vary how that sentence is built: do not default to '<name> is a <age>-year-old ...' or "
    "'Now <age>, ...'. Do not mention his birthday.\n"
    "5. report_text is 120 to 180 words of continuous prose: no headings, lists, or markdown.\n"
    "6. Judge him the way your club would: what he would add, what the numbers suggest, and what "
    "you would want to see next. Use the match detail (starts, when and how he scores, who he "
    "combines with, results with and without him) to say something specific about how he plays. "
    "Opinions are fine; invented facts are not.\n"
    "7. recommendation is SIGN, MONITOR or PASS and must agree with the report. headline is a "
    "plain summary of under 12 words.\n"
    "8. Write as a scout who has watched him. Never mention the facts, notes, data, coverage, or "
    "which competitions are included, and do not say 'all competitions'.\n"
    "9. Mention height, footedness or discipline only when it matters to your judgment of him."
)

GUIDE = {
    "later_signer": "Your club did not necessarily face him this season; you have followed him from the stands "
                    "and on video. Assess whether he would strengthen your squad.",
    "european_opponent": "Your club met him in European competition this season. Draw on those meetings as well "
                         "as his season numbers.",
    "same_league": "Your club plays in the same league as he does. Draw on your league meetings as well as his "
                   "season numbers.",
}
GUIDE_DOM = ("Your club plays in the same domestic league as his club, but those league matches are not "
             "in your notes. Judge him from his European record and the numbers you have.")

SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "headline": {"type": "STRING"},
        "report_text": {"type": "STRING"},
        "recommendation": {"type": "STRING", "enum": ["SIGN", "MONITOR", "PASS"]},
    },
    "required": ["headline", "report_text", "recommendation"],
    "propertyOrdering": ["headline", "report_text", "recommendation"],
}


def user_prompt(rep):
    guide = GUIDE_DOM if (rep["perspective"] == "same_league" and rep["league_src"] == "dom") else GUIDE[rep["perspective"]]
    return (f"You scout for {CLUB_NAME.get(rep['scout_club_id'], rep['scout_club_id'])}. {guide}\n\n"
            f"FACTS\n{rep['facts']}\n\n"
            "Write the report now as JSON with headline, report_text and recommendation.")


R["prompt"] = [user_prompt(rep) for rep in R.to_dict("records")]
EST_IN = (len(SYSTEM) + R.prompt.str.len()) / 4
print(f"system instruction {len(SYSTEM):,} chars; prompt median {R.prompt.str.len().median():,.0f} chars")
print(f"estimated input tokens per report ~{EST_IN.median():,.0f} (chars/4)")
hb("6 · The prompt", f"{PROMPT_VERSION}; est. input ~{EST_IN.median():,.0f} tok/report (chars/4)")

---
## 7 · QA rules

The same checks run on the sample, the full run and every repair. **Hard** failures are
regenerated. **Soft** flags are counted and reported for a human look.

| Kind | Check |
| :-- | :-- |
| hard | response missing, unparseable, or a field empty; recommendation not SIGN/MONITOR/PASS |
| hard | `report_text` under 90 or over 230 words |
| hard | the player's full name not in the first 300 characters (accent-insensitive) |
| hard | a year later than the season's end, or a club he joined later named in the text |
| hard | garbled characters (a control character, or a symbol inside a word such as "Iv!n") |
| hard | the words *injury*/*injured*/*suspension*/*recovery*, birthday wording ("birthday", "turned 23", "reaches 21"), or markdown (`**`, `#`, bullet lines) |
| hard | prompt plumbing in the prose (Part 2's check, adapted): "covered matches", "the facts", "dataset", "as an AI", "CymbalGoal", "all competitions", and similar; or any `[ ] { }` bracket |
| soft | `report_text` outside 120–180 words |
| soft | a number in the text (4 or more) that appears nowhere in the fact block |
| soft | a proper name of two or more words (a person, stadium or place) that appears nowhere in the fact block, so outside knowledge can be checked by eye |

In [ ]:
_INT_RE = re.compile(r"(?<![\d.,])\d{1,3}(?:,\d{3})+(?!\d|,\d)|(?<![\d.,])\d+(?!\d|[.,]\d)")
_YEAR_RE = re.compile(r"\b(19[89]\d|20[0-4]\d)\b")
_BAD_WORDS = re.compile(r"\b(injur\w*|suspen\w*|(?:full|a) recovery|recovery from|recover(?:s|ed|ing) from)\b", re.I)
_GARBLED = re.compile(r"[\x00-\x08\x0b-\x1f\x7f\ufffd]|[^\W\d_][!?#$%^*|~\\@][^\W\d_]")
_MD = re.compile(r"(\*\*|^#|^\s*[-*•]\s)", re.M)
_PLUMBING = re.compile(
    r"(?i)(?:as an ai|language model|this prompt|fact block|cymbalgoal|the snapshot"
    r"|(?:this|the|our|provided|given|supplied)\s+(?:data\s?set|dataset|facts|notes|data)\b"
    r"|covered\s+(?:\w+\s+){0,2}(?:matches|competitions|leagues|games)"
    r"|(?:matches|competitions|leagues)\s+(?:covered|included|listed)"
    r"|(?:big 5|big five)\s+leagues?,?\s+(?:uefa|champions)\b"
    r"|all competitions)")
_BIRTHDAY = re.compile(r"(?i)\bbirthday\b|\bturn(?:ed|s|ing) \d{2}\b"
                       r"|\breach(?:ed|es|ing)? (?:the age of )?\d{2}\b(?! (?:goals?|assists?|appearances?|matches|games|"
                       r"starts|minutes|league|clean|yellow|cards?|times)\b)")
_FUT = defaultdict(list)
for r in _fut.itertuples(index=False):
    _FUT[int(r.player_id)].append((r.transfer_date, str(r.to_club_name)))


_LEAD = set("a an the his her their our we he they this that these those when while with without although though "
            "despite across against now having given for at on as in of to by from after before during both even "
            "still yet but and or so if since until overall crucially however meanwhile ultimately moreover "
            "additionally instead there here what which who where its it once whether only also under over "
            "between".split())
_NAME_RUN = re.compile(r"[A-ZÀ-ÖØ-Þ][\w’'-]*(?:[ \t]+(?:(?:de|da|di|do|dos|del|della|van|von|der|den|le|la|el|al|y)[ \t]+)*"
                       r"[A-ZÀ-ÖØ-Þ][\w’'-]*)+")


def unknown_names(text, facts):
    known = set(re.findall(r"\w+", fold(facts)))
    out = []
    for m in _NAME_RUN.finditer(text):
        words = m.group(0).split()
        _toks = lambda w: re.findall(r"\w+", fold(re.sub(r"[’']s$", "", w)))
        at_start = not text[:m.start()].strip() or text[:m.start()].rstrip()[-1:] in ".!?:;"
        # a capital at the start of a sentence is not evidence of a name ("Elite Galatasaray")
        if words[0].lower() in _LEAD or (at_start and not all(t in known for t in _toks(words[0]))):
            words = words[1:]
        caps = [w for w in words if w[:1].isupper()]
        if len(caps) < 2:
            continue
        if all(t in known for w in caps for t in _toks(w)):
            continue
        out.append(" ".join(words).rstrip(".’'"))
    return out


def nums(text):
    return {int(x.replace(",", "")) for x in _INT_RE.findall(text)}


def qa(rep, out):
    hard, soft = [], []
    if not out or out.get("error"):
        return ["no response: " + str((out or {}).get("error", ""))[:80]], soft
    txt, head, rec = out.get("report_text") or "", out.get("headline") or "", out.get("recommendation") or ""
    if not txt.strip() or not head.strip():
        hard.append("empty field")
    if rec not in ("SIGN", "MONITOR", "PASS"):
        hard.append(f"recommendation {rec!r}")
    w = len(txt.split())
    if not HARD_WORDS[0] <= w <= HARD_WORDS[1]:
        hard.append(f"{w} words")
    elif not REPORT_WORDS[0] <= w <= REPORT_WORDS[1]:
        soft.append(f"{w} words")
    if fold(rep["player_name"]) not in fold(txt[:300]):
        hard.append("name not in opening")
    end_year = rep["season"] + 1
    yrs = [int(y) for y in _YEAR_RE.findall(txt + " " + head) if int(y) > end_year]
    if yrs:
        hard.append(f"future year {yrs[0]}")
    season_end = pd.Timestamp(f"{end_year}-06-30")
    known = fold(rep["facts"])
    for d, club in _FUT.get(rep["player_id"], []):
        if d > season_end and len(club) > 3 and fold(club) not in known and fold(club) in fold(txt):
            hard.append(f"later club named: {club}")
            break
    if _BAD_WORDS.search(txt + " " + head):
        hard.append("injury/suspension wording")
    if _MD.search(txt):
        hard.append("markdown")
    if _BIRTHDAY.search(txt + " " + head):
        hard.append("birthday wording")
    m = _PLUMBING.search(txt + " " + head)
    if m:
        hard.append(f"plumbing: {m.group(0)!r}")
    if re.search(r"[\[\]{}]", txt + head):
        hard.append("brackets")
    if _GARBLED.search(txt + head):
        hard.append("garbled characters")
    odd = unknown_names(txt + " . " + head, rep["facts"])
    if odd:
        soft.append(f"names not in facts: {odd[:4]}")
    stray = sorted(n for n in nums(txt) - nums(rep["facts"]) if n >= 4 and not (1900 <= n <= end_year))
    if stray:
        soft.append(f"numbers not in facts: {stray[:4]}")
    return hard, soft


print("QA rules ready")

---
## 8 · Generation machinery and the 25-report sample

The generation code is Part 2's, carried over as it is:

- **`Meter`** counts tokens, calls, retries, failures and 429s across threads. On Gemini 3.x the
  output count already includes reasoning, so reasoning is shown but never added on top.
- **`Throttle`** sizes the pool every 30 seconds: more than 15% of calls rate-limited cuts the
  workers to 60%, and a window under 3% lets them grow, up to `MAX_WORKERS_CEILING`.
- **`_backoff`** is exponential with jitter inside the cap, uses the server's own "retry in N
  seconds" hint when there is one, and never waits less than 20 s after a 429. Each report gets
  a 120 s retry budget in total.
- **`_Fatal`** stops retries that can't succeed: a response cut off at `MAX_OUTPUT_TOKENS`, or one
  blocked by the model.

The sample is picked to cover the cases most likely to go wrong: the most valuable players,
extended absences, shared names (including a Paulinho), UEFA-only player-seasons, mid-season
movers, and all three perspectives. It is cached in the bucket, so later runs reuse it.

**Run 1 stops being useful here.** Read the reports in the handback before setting
`RUN_FULL = True`.

In [ ]:
S = "8 · Sample"
RETRYABLE_CODES = {408, 409, 425, 429, 500, 502, 503, 504}
_RETRY_AFTER_RE = re.compile(r"(?:retry|try again)\s+(?:in|after)\s+([\d.]+)\s*(s|sec|second)", re.I)


def _status_code(exc):
    for attr in ("code", "status_code"):
        v = getattr(exc, attr, None)
        if isinstance(v, int):
            return v
    m = re.search(r"\b(4\d\d|5\d\d)\b", str(exc))
    return int(m.group(1)) if m else None


def _suggested_delay(exc):
    for attr in ("response", "_response"):
        hdrs = getattr(getattr(exc, attr, None), "headers", None)
        if hdrs and hasattr(hdrs, "get"):
            for key in ("retry-after", "Retry-After", "x-retry-after"):
                v = hdrs.get(key)
                if v:
                    try:
                        return float(v)
                    except (TypeError, ValueError):
                        pass
    m = _RETRY_AFTER_RE.search(str(exc))
    return float(m.group(1)) if m else None


def _backoff(attempt, code=None, exc=None):
    jitter = 0.5 + random.random()
    if exc is not None:
        hinted = _suggested_delay(exc)
        if hinted:
            return min(BACKOFF_CAP_S, hinted * jitter + hinted * 0.5)
    wait = (BACKOFF_BASE ** attempt) * jitter
    if code == 429:
        wait = max(RATE_LIMIT_MIN_WAIT_S * jitter, wait)
    return min(BACKOFF_CAP_S, wait)


class _Fatal(Exception):
    pass


class Meter:
    def __init__(self):
        self._lock = threading.Lock()
        self.in_tokens = self.out_tokens = self.thinking_tokens = 0
        self.calls = self.retries = self.failures = self.rate_limited = 0

    def add(self, *, ti=0, to=0, tt=0, retries=0, failed=False, rate_limited=0):
        with self._lock:
            self.in_tokens += ti; self.out_tokens += to; self.thinking_tokens += tt
            self.calls += 1; self.retries += retries; self.failures += int(failed)
            self.rate_limited += rate_limited

    def cost(self):
        # out_tokens already includes reasoning on Gemini 3.x (Part 2); not added again.
        return (self.in_tokens * PRICE_IN_PER_M + self.out_tokens * PRICE_OUT_PER_M) / 1e6


class Throttle:
    def __init__(self, start, floor=MIN_WORKERS, ceiling=None):
        self.start, self.floor, self.workers = start, floor, start
        self.ceiling = ceiling or start
        self._seen_calls = self._seen_429 = 0

    def adjust(self, meter):
        calls = meter.calls - self._seen_calls
        limited = meter.rate_limited - self._seen_429
        self._seen_calls, self._seen_429 = meter.calls, meter.rate_limited
        if calls <= 0:
            return self.workers
        rate = limited / calls
        before = self.workers
        if rate > THROTTLE_TRIGGER_RATE:
            self.workers = max(self.floor, int(self.workers * 0.6))
        elif rate < THROTTLE_GROW_BELOW and self.workers < self.ceiling:
            self.workers = min(self.ceiling, int(self.workers * 1.5) + 1)
        if self.workers != before:
            print(f"    throttle: {rate:.0%} rate-limited -> workers {before} -> {self.workers}")
        return self.workers


SCHEMA_SDK = {k: v for k, v in SCHEMA.items() if k != "propertyOrdering"}
SCHEMA_SDK["property_ordering"] = SCHEMA["propertyOrdering"]


def gen_config(seed=SEED):
    kw = dict(temperature=TEMPERATURE, seed=seed, max_output_tokens=MAX_OUTPUT_TOKENS,
              system_instruction=SYSTEM, response_mime_type="application/json", response_schema=SCHEMA_SDK, **afc_off())
    tc = thinking_config()
    if tc is not None:
        kw["thinking_config"] = tc
    return gtypes.GenerateContentConfig(**kw)


def parse_response_text(text):
    d = json.loads(text)
    out = {k: d.get(k) for k in ("headline", "report_text", "recommendation")}
    return {k: (v.strip() if isinstance(v, str) else v) for k, v in out.items()}


GEN_CONFIG = None


def generate_one(rep, meter):
    global GEN_CONFIG
    if GEN_CONFIG is None:
        GEN_CONFIG = gen_config()
    last, n_429 = "", 0
    deadline = time.time() + RETRY_BUDGET_S
    for attempt in range(MAX_RETRIES):
        try:
            # a repair gets a fresh seed: with the fixed SEED and the same prompt, Gemini tends to write
            # the same report again, failure included
            cfg = gen_config(seed=random.randint(1, 2**31 - 1)) if rep.get("repair") else GEN_CONFIG
            resp = client(GEN_LOCATION).models.generate_content(model=GEN_MODEL, contents=rep["prompt"], config=cfg)
            cand = (getattr(resp, "candidates", None) or [None])[0]
            finish = str(getattr(cand, "finish_reason", "") or "")
            if "MAX_TOKENS" in finish.upper():
                raise _Fatal(f"hit MAX_OUTPUT_TOKENS ({MAX_OUTPUT_TOKENS:,})")
            if "SAFETY" in finish.upper() or "RECITATION" in finish.upper():
                raise _Fatal(f"blocked by the model, finish_reason={finish}")
            text = (resp.text or "").strip()
            if not text:
                raise RuntimeError(f"empty response body (finish_reason={finish or 'unknown'})")
            out = parse_response_text(text)            # a JSON error is retried like any other
            um = getattr(resp, "usage_metadata", None)
            ti = int(getattr(um, "prompt_token_count", 0) or 0)
            to = int(getattr(um, "candidates_token_count", 0) or 0)
            tt = int(getattr(um, "thoughts_token_count", 0) or 0)
            meter.add(ti=ti, to=to, tt=tt, retries=attempt, rate_limited=n_429)
            return {"report_id": rep["report_id"], **out, "in_tok": ti, "out_tok": to, "think_tok": tt,
                    "attempts": attempt + 1, "error": ""}
        except _Fatal as exc:
            last = f"Fatal: {exc}"
            break
        except Exception as exc:  # noqa: BLE001
            code = _status_code(exc)
            last = f"{type(exc).__name__} {code or ''}: {str(exc)[:200]}"
            if code == 429:
                n_429 += 1
            if code is not None and code not in RETRYABLE_CODES and 400 <= code < 500:
                break
            if attempt < MAX_RETRIES - 1:
                wait = _backoff(attempt, code, exc)
                if time.time() + wait > deadline:
                    last += f" | gave up: {RETRY_BUDGET_S:.0f}s retry budget exhausted"
                    break
                time.sleep(wait)
    meter.add(retries=MAX_RETRIES, failed=True, rate_limited=n_429)
    return {"report_id": rep["report_id"], "headline": None, "report_text": None, "recommendation": None,
            "in_tok": 0, "out_tok": 0, "think_tok": 0, "attempts": MAX_RETRIES, "error": last or "failed"}


def pick_sample():
    v = R.player_id.map(PL.highest_market_value_in_eur).fillna(0)
    picks = []

    def take(df, n):
        for rid in df.report_id.tolist():
            if n <= 0:
                break
            if rid not in picks:
                picks.append(rid); n -= 1
    take(R.assign(v=v).sort_values(["v", "season"], ascending=False).drop_duplicates("player_id"), 5)
    take(R[R.apply(lambda r: (r.player_id, r.season) in ABS, axis=1)].sample(frac=1, random_state=SEED), 4)
    take(R[R.player_name.map(fold) == "paulinho"], 1)
    shared = R.player_name.map(R.drop_duplicates("player_id").player_name.value_counts()) > 1
    take(R[shared & (R.player_name.map(fold) != "paulinho")].sample(frac=1, random_state=SEED), 2)
    take(R[R.league_src == "dom"].sample(frac=1, random_state=SEED), 3)
    take(R[R.apply(lambda r: len(PCL.get((r.player_id, r.season), [])) > 1, axis=1)].sample(frac=1, random_state=SEED), 2)
    for persp in ("later_signer", "european_opponent", "same_league"):
        take(R[R.perspective == persp].sample(frac=1, random_state=SEED), 2)
    take(R.sample(frac=1, random_state=SEED), SAMPLE_N - len(picks))
    return picks[:SAMPLE_N]


SAMPLE_URI = f"{GCS_WORK}/samples/sample-{PROMPT_VERSION}.jsonl"
SAMPLE = []
if not RUN_SAMPLE:
    print("skipped—RUN_SAMPLE is False")
    hb(S, "skipped: RUN_SAMPLE is False")
elif not PREFLIGHT_OK:
    print("skipped—preflight failed")
    hb(S, "skipped: preflight failed (see section 2)")
else:
    cached = None if RERUN_SAMPLE else get_text(SAMPLE_URI)
    if cached:
        SAMPLE = [json.loads(l) for l in cached.splitlines() if l.strip()]
        print(f"reusing the cached sample ({len(SAMPLE)} reports) from {SAMPLE_URI}")
    else:
        reps = R.set_index("report_id").loc[pick_sample()].reset_index().to_dict("records")
        meter = Meter()
        with ThreadPoolExecutor(max_workers=SAMPLE_WORKERS) as pool:
            outs = {o["report_id"]: o for o in pool.map(lambda r: generate_one(r, meter), reps)}
        for rep in reps:
            o = outs[rep["report_id"]]
            hard, soft = qa(rep, o)
            SAMPLE.append({"report_id": rep["report_id"], "player": rep["player_name"], "season": rep["season"],
                           "perspective": rep["perspective"],
                           "scout": CLUB_NAME.get(rep["scout_club_id"], rep["scout_club_id"]),
                           "facts": rep["facts"], **o, "qa_hard": hard, "qa_soft": soft})
        p = BASE / "results" / f"sample-{PROMPT_VERSION}.jsonl"
        p.write_text("\n".join(json.dumps(x, default=str, ensure_ascii=False) for x in SAMPLE) + "\n")
        put(p, SAMPLE_URI)
        print(f"sample saved to {SAMPLE_URI}  (cost ≈ ${meter.cost():.3f}, {meter.rate_limited} rate-limited calls)")

if SAMPLE:
    n_hard = sum(1 for x in SAMPLE if x["qa_hard"])
    n_soft = sum(1 for x in SAMPLE if x["qa_soft"])
    words = [len((x.get("report_text") or "").split()) for x in SAMPLE]
    recs = Counter(x.get("recommendation") for x in SAMPLE)
    tin = np.mean([x.get("in_tok", 0) for x in SAMPLE]); tout = np.mean([x.get("out_tok", 0) for x in SAMPLE])
    tth = np.mean([x.get("think_tok", 0) for x in SAMPLE])
    per_report = (tin * PRICE_IN_PER_M + tout * PRICE_OUT_PER_M) / 1e6
    print(f"\nQA: {n_hard} hard, {n_soft} soft of {len(SAMPLE)}; words median {int(np.median(words))} "
          f"(min {min(words)}, max {max(words)}); recommendations {dict(recs)}")
    print(f"tokens/report: in {tin:,.0f}, out {tout:,.0f} (of which reasoning {tth:,.0f})")
    print(f"projected full run: ${per_report * len(R):,.0f} for {len(R):,} reports at ${per_report * 1000:.2f}/1,000")
    for x in SAMPLE[:3]:
        print(f"\n— {x['player']} {season_label(x['season'])}, {x['perspective']} by {x['scout']} "
              f"[{x.get('recommendation')}] {x.get('headline')}\n{x.get('report_text') or x.get('error')}")
    _formula = sum(1 for x in SAMPLE if re.match(r"^[^.]{0,60}\bis an? (?:\w+-)?\d+-year-old", x.get("report_text") or ""))
    _height = sum(1 for x in SAMPLE if re.search(r"\b\d{3}\s?cm\b", x.get("report_text") or ""))
    _now = sum(1 for x in SAMPLE if re.match(r"^Now (?:aged )?\d+", x.get("report_text") or ""))
    _odd = [f for x in SAMPLE for f in (x.get("qa_soft") or []) if f.startswith("names not in facts")]
    print(f"reports naming something not in the facts: {len(_odd)}/{len(SAMPLE)}")
    hb(S, f"names not in facts: {len(_odd)}/{len(SAMPLE)} " + "; ".join(o[len('names not in facts: '):] for o in _odd))
    print(f"openings '<name> is a N-year-old': {_formula}/{len(SAMPLE)}; 'Now N': {_now}/{len(SAMPLE)}; "
          f"height mentioned: {_height}/{len(SAMPLE)}")
    hb(S, f"style: '<name> is a N-year-old' openings {_formula}/{len(SAMPLE)}; 'Now N' openings {_now}/{len(SAMPLE)}; "
          f"height mentioned {_height}/{len(SAMPLE)}")
    hb(S, f"{len(SAMPLE)} reports; QA hard {n_hard}, soft {n_soft}; words median {int(np.median(words))} "
          f"({min(words)}–{max(words)}); recs {dict(recs)}; tok in {tin:,.0f} out {tout:,.0f} (reasoning {tth:,.0f}); "
          f"projected ${per_report * len(R):,.0f} for {len(R):,}")

---
## 9 · Full generation

Runs only with `RUN_FULL = True`. Part 2's pieces (`Meter`, `Throttle`, backoff, checkpoints)
in a continuous pool: a slot is refilled the moment a call returns. Part 2 sent fixed batches of
100 and waited for the slowest call in each, so one report sitting out a 20-second 429 backoff
held up 99 finished ones; at full scale that capped the run near 100 reports a minute.

**Checkpoints.** Every `SAVE_EVERY_N_REPORTS` finished reports the results so far are written to
the runtime's disk (a count kept by this run, never `len(results) % N`, which silently stops
saving on a resume), and at most every 2 minutes the checkpoint is mirrored to
`checkpoints/gen-{prompt version}.parquet` in the bucket. Failed reports stay in the checkpoint
with their error, so the next run retries exactly those.

**Resuming.** If the runtime dies, or you stop the cell, just Run all again. The checkpoint is
restored (local first, then the bucket) and only the missing or failed reports are generated.

**The cold-start guard.** If the bucket listing *errors* (as opposed to finding nothing), this
section refuses to start. A failed restore and a first run are otherwise the same code path,
with the whole corpus's cost between them.

In [ ]:
S = "9 · Full generation"
CKPT_URI = f"{GCS_WORK}/checkpoints/gen-{PROMPT_VERSION}.parquet"
CKPT = BASE / "results" / f"gen-{PROMPT_VERSION}.parquet"
GEN_COLS = ["report_id", "headline", "report_text", "recommendation", "in_tok", "out_tok", "think_tok",
            "attempts", "error"]
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def restore_checkpoint():
    rows = {}
    sources = []
    if CKPT.exists():
        d = pd.read_parquet(CKPT)
        rows = {r["report_id"]: r for r in d.to_dict("records")}
        sources.append(f"local ({len(rows):,})")
    r = gcs("ls", CKPT_URI)
    if r.returncode == 0:
        b = gcs("cat", CKPT_URI, binary=True)
        if b.returncode != 0:
            return rows, "ERROR", f"listed but unreadable: {b.stderr[:200]!r}"
        d = pd.read_parquet(io.BytesIO(b.stdout))
        sources.append(f"bucket ({len(d):,})")
        if len(d) > len(rows):
            rows = {x["report_id"]: x for x in d.to_dict("records")}
            d.to_parquet(CKPT, index=False)
    elif "matched no objects" not in r.stderr.lower() and "no urls matched" not in r.stderr.lower():
        return rows, "ERROR", r.stderr.strip()[:300]
    return rows, ("none" if not sources else " + ".join(sources)), ""


def save_checkpoint(push):
    pd.DataFrame([{k: GEN[rid].get(k) for k in GEN_COLS} for rid in GEN]).to_parquet(CKPT, index=False)
    if push:
        put(CKPT, CKPT_URI)


def run_generation(reps, label):
    # Part 2's pieces without the batch barrier: a slot is refilled as soon as a call returns, so a
    # report waiting out a 429 holds one slot, not a whole batch of finished ones.
    if not reps:
        return Meter()
    meter = Meter()
    throttle = Throttle(MAX_WORKERS, ceiling=MAX_WORKERS_CEILING)
    queue = list(reversed(reps))
    inflight = set()
    t0 = last_push = last_adj = time.time()
    done = since_save = 0
    saved = {"local": 0, "bucket": 0, "at": "-"}
    bar = tqdm(total=len(reps), desc=label, unit="report") if tqdm else None
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS_CEILING)

    def status():
        el = max(time.time() - t0, 1)
        per_min = done * 60 / el
        eta = (len(reps) - done) / max(done / el, 1e-9) / 60
        return (f"{done:,}/{len(reps):,}, {per_min:,.0f}/min, ETA {eta:,.0f} min, workers {throttle.workers}, "
                f"429s {meter.rate_limited}, failures {meter.failures}, ≈${meter.cost():,.2f}")

    try:
        while queue or inflight:
            while queue and len(inflight) < throttle.workers:
                inflight.add(pool.submit(generate_one, queue.pop(), meter))
            finished, inflight = fut_wait(inflight, timeout=5, return_when=FIRST_COMPLETED)
            for f in finished:
                o = f.result()
                GEN[o["report_id"]] = o
                done += 1
                since_save += 1
                if bar:
                    bar.update(1)
            now = time.time()
            if now - last_adj >= THROTTLE_WINDOW_S:
                throttle.adjust(meter)
                last_adj = now
                if bar:
                    bar.set_postfix_str(f"workers {throttle.workers}, 429s {meter.rate_limited}, "
                                        f"fail {meter.failures}, ≈${meter.cost():,.0f}, saved {saved['local']:,} "
                                        f"(bucket {saved['bucket']:,} at {saved['at']})")
                else:
                    print(f"  {label}: {status()}")
            if since_save >= SAVE_EVERY_N_REPORTS and (queue or inflight):
                push = now - last_push >= GCS_PUSH_EVERY_S
                save_checkpoint(push)
                since_save = 0
                saved["local"] = len(GEN)
                if push:
                    last_push = now
                    saved["bucket"], saved["at"] = len(GEN), time.strftime("%H:%M")
                    print(f"  {label}: {status()}, checkpoint pushed to bucket")
    except KeyboardInterrupt:
        print(f"\n⏹  stopped: {done:,} reports finished in this run are being saved; Run all again to resume")
        raise
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
        if done:
            save_checkpoint(True)           # never lose finished work, even when the cell is stopped
        if bar:
            bar.close()
    el = time.time() - t0
    print(f"{label}: {len(reps):,} in {el / 60:.1f} min ({len(reps) * 60 / max(el, 1):,.0f}/min); tokens in {meter.in_tokens:,} "
          f"out {meter.out_tokens:,} (reasoning {meter.thinking_tokens:,}); retries {meter.retries:,}, "
          f"failures {meter.failures:,}, 429s {meter.rate_limited:,}; ≈${meter.cost():,.2f}")
    if meter.rate_limited:
        print(f"   throttle ended at {throttle.workers} workers (ceiling {MAX_WORKERS_CEILING}). Sustained 429s mean "
              "the quota is the ceiling; the throttle has already found it.")
    return meter


GEN, CKPT_STATUS, CKPT_ERR = restore_checkpoint()
n_ok = sum(1 for o in GEN.values() if not o.get("error"))
print(f"checkpoint: {CKPT_STATUS}; {n_ok:,} of {len(R):,} reports generated, "
      f"{len(GEN) - n_ok:,} failed and waiting for a retry")
if not RUN_FULL:
    print("full generation skipped—RUN_FULL is False")
elif not PREFLIGHT_OK:
    print("full generation skipped—preflight failed")
elif CKPT_STATUS == "ERROR":
    print(f"🔴 REFUSING TO START: the checkpoint could not be read ({CKPT_ERR}).\n"
          "   A failed restore looks exactly like a first run. Fix bucket access and rerun.")
else:
    todo = [rep for rep in R.to_dict("records") if rep["report_id"] not in GEN or GEN[rep["report_id"]].get("error")]
    print(f"to generate: {len(todo):,}")
    if todo:
        run_generation(todo, "generate")
n_ok = sum(1 for o in GEN.values() if not o.get("error"))
GEN_COMPLETE = n_ok == len(R)
hb(S, f"checkpoint {CKPT_STATUS}; generated {n_ok:,} of {len(R):,}; failed {len(GEN) - n_ok:,}; complete={GEN_COMPLETE}")

---
## 10 · QA and repair

Runs once every report has been generated. The QA rules from section 7 run over all of them.
Every report with a **hard** failure is regenerated through the same loop (up to
`REPAIR_ROUNDS` rounds, and never more than `REPAIR_LIMIT` reports at once), and QA runs again.
The final, QA'd results are saved to `results/reports-{prompt version}.parquet` in the bucket.

In [ ]:
S = "10 · QA and repair"
RESULTS_URI = f"{GCS_WORK}/results/reports-{PROMPT_VERSION}.parquet"
RES, QA_CLEAN = None, False
REPS = {rep["report_id"]: rep for rep in R.to_dict("records")}


_FIX = {
    "name not in opening": "The first sentence must name the player exactly as written in the facts: {name}.",
    "birthday wording": "Give his age plainly ('the 27-year-old'); never say when he turned or will turn an age.",
    "injury/suspension wording": "Never use the words injury, injured, suspension or recovery; describe gaps only as absences.",
    "plumbing": "Never mention the facts, notes, data, coverage or 'all competitions'.",
    "future year": "Mention nothing after the report date.",
    "later club named": "Mention nothing after the report date, including clubs he joined later.",
    "garbled characters": "Spell every name correctly, with its accents.",
    "markdown": "Write plain continuous prose: no markdown, headings or lists.",
    "brackets": "Do not use brackets or braces.",
    "empty field": "Fill in headline, report_text and recommendation.",
}


def repair_note(hard, rep):
    # Tell the model what QA rejected, so a repair is a correction and not just a reroll.
    fixes = []
    for h in hard:
        key = h.split(":")[0]
        if key.endswith("words"):
            fixes.append(f"report_text must be {REPORT_WORDS[0]} to {REPORT_WORDS[1]} words (the last draft was {key}).")
        elif key in _FIX:
            fixes.append(_FIX[key].format(name=rep["player_name"]))
        else:
            fixes.append(f"Avoid this problem: {h[:120]}.")
    return ("\n\nA previous draft of this report was rejected. Write a new one that keeps every rule and fixes this:\n- "
            + "\n- ".join(dict.fromkeys(fixes)))


def qa_all(ids):
    out = {}
    for rid in ids:
        h, s_ = qa(REPS[rid], GEN.get(rid))
        out[rid] = (h, s_)
    return out


if not GEN_COMPLETE:
    print(f"waiting—{sum(1 for o in GEN.values() if not o.get('error')):,} of {len(R):,} generated")
else:
    t0 = time.time()
    QA = qa_all(R.report_id)
    repaired = set()
    for rnd in range(1, REPAIR_ROUNDS + 1):
        need = [rid for rid, (h, _) in QA.items() if h]
        _why = Counter(x.split(":")[0] for rid in need for x in QA[rid][0])
        print(f"QA pass {rnd}: {len(need):,} hard failures {dict(_why)}")
        hb(S, f"QA pass {rnd}: {len(need):,} hard failures {dict(_why)}")
        if not need or not REPAIR_ONLINE or not PREFLIGHT_OK:
            break
        if len(need) > REPAIR_LIMIT:
            print(f"🔴 {len(need):,} hard failures is over REPAIR_LIMIT ({REPAIR_LIMIT:,}); not repairing. "
                  "Read the reasons in the handback first: a pattern this large is a prompt problem.")
            break
        if rnd == 1:
            _ex = defaultdict(list)
            for rid in need:
                for x in QA[rid][0]:
                    if len(_ex[x.split(":")[0]]) < 3:
                        _ex[x.split(":")[0]].append(f"{rid}: {x[:120]}")
            for k, v in _ex.items():
                hb(S, f"  e.g. {k}: " + " | ".join(v))
        run_generation([{**REPS[r], "prompt": REPS[r]["prompt"] + repair_note(QA[r][0], REPS[r]), "repair": True}
                        for r in need], f"repair {rnd}")
        repaired |= set(need)
        QA.update(qa_all(need))
    rows = []
    for rid in R.report_id:
        o = GEN[rid]
        h, s_ = QA[rid]
        rows.append({**{k: o.get(k) for k in GEN_COLS}, "source": "repair" if rid in repaired else "full",
                     "qa_hard": "; ".join(h), "qa_soft": "; ".join(s_)})
    RES = pd.DataFrame(rows)
    _rp = BASE / "results" / f"reports-{PROMPT_VERSION}.parquet"
    RES.to_parquet(_rp, index=False)
    put(_rp, RESULTS_URI)
    hard_left = RES[RES.qa_hard != ""]
    QA_CLEAN = len(hard_left) == 0
    reasons = Counter(r.split(":")[0] for x in RES.qa_hard for r in x.split("; ") if r)
    sreasons = Counter(r.split(":")[0] for x in RES.qa_soft for r in x.split("; ") if r)
    wc = RES.report_text.fillna("").str.split().str.len()
    tin, tout, tth = (pd.to_numeric(RES[c], errors="coerce").fillna(0) for c in ("in_tok", "out_tok", "think_tok"))
    cost = (tin.sum() * PRICE_IN_PER_M + tout.sum() * PRICE_OUT_PER_M) / 1e6
    print(f"\nremaining hard failures: {len(hard_left):,}  {dict(reasons)}")
    print(f"soft flags: {dict(sreasons)}")
    print(f"words: median {wc.median():.0f}, 5th pct {wc.quantile(.05):.0f}, 95th {wc.quantile(.95):.0f}")
    print(f"recommendations: {RES.recommendation.value_counts().to_dict()}")
    print(f"repaired: {len(repaired):,}; tokens in {tin.sum() / 1e6:,.1f}M, out {tout.sum() / 1e6:,.1f}M "
          f"(reasoning {tth.sum() / 1e6:,.2f}M); cost of the kept reports ≈ ${cost:,.2f}")
    hb(S, f"results {len(RES):,}; repaired {len(repaired):,}; hard left {len(hard_left):,} {dict(reasons)}; soft {dict(sreasons)}")
    hb(S, f"words median {wc.median():.0f} (p5 {wc.quantile(.05):.0f}, p95 {wc.quantile(.95):.0f}); "
          f"recs {RES.recommendation.value_counts().to_dict()}")
    hb(S, f"tokens in {tin.sum() / 1e6:,.1f}M out {tout.sum() / 1e6:,.1f}M (reasoning {tth.sum() / 1e6:,.2f}M); ≈${cost:,.2f}")
    for _, x in hard_left.head(8).iterrows():
        hb(S, f"  still failing {x.report_id}: {x.qa_hard}")
    _soft_ex = RES[RES.qa_soft != ""].head(5)
    for _, x in _soft_ex.iterrows():
        hb(S, f"  soft {x.report_id}: {x.qa_soft}")
    # corpus-wide style and outside-knowledge checks, the same ones the sample reports
    _txt = RES.report_text.fillna("")
    _n = max(len(RES), 1)
    _form = _txt.str.match(r"^[^.]{0,60}\bis an? (?:\w+-)?\d+-year-old").sum()
    _now = _txt.str.match(r"^Now (?:aged )?\d+").sum()
    _ht = _txt.str.contains(r"\b\d{3}\s?cm\b", regex=True).sum()
    hb(S, f"style across the corpus: '<name> is a N-year-old' {_form / _n:.1%}; 'Now N' {_now / _n:.1%}; "
          f"height mentioned {_ht / _n:.1%}")
    import ast
    _odd = Counter(nm for x in RES.qa_soft for r in x.split("; ") if r.startswith("names not in facts")
                   for nm in ast.literal_eval(r[len("names not in facts: "):]))
    _n_odd = sum(1 for x in RES.qa_soft if "names not in facts" in x)
    hb(S, f"names not in facts: {_n_odd:,} reports ({_n_odd / _n:.1%}); most common: " +
          "; ".join(f"{k} ({v})" for k, v in _odd.most_common(25)))
    _pr = RES.merge(R[["report_id", "perspective"]], on="report_id", how="left")
    hb(S, "recs by perspective: " + "; ".join(
        f"{k} {v.recommendation.value_counts().to_dict()}" for k, v in _pr.groupby("perspective")))
    for _, x in RES.sample(n=min(6, len(RES)), random_state=7).iterrows():
        hb(S, f"  random {x.report_id} [{x.recommendation}] {x.headline} :: {x.report_text}")

---
## 11 · Embeddings

`gemini-embedding-001` at 3072 dims, `RETRIEVAL_DOCUMENT`, on `report_text` only, so the stored
column is exactly the text that was embedded. It uses the same backoff and `Throttle` as
generation (the pool is resized between shards), and writes
**shards** of 2,000 rows to `emb/` in the bucket as it goes. A dead runtime loses at most one
unfinished shard; a rerun restores the finished ones and carries on.

It runs only when every report has passed hard QA.

In [ ]:
S = "11 · Embeddings"
EMB_PREFIX = f"{GCS_WORK}/emb/{PROMPT_VERSION}"
EMB = {}


def embed_one(rid, text, meter):
    cfg = gtypes.EmbedContentConfig(output_dimensionality=EMBED_DIM, task_type=EMBED_TASK_TYPE)
    last, n_429 = "", 0
    deadline = time.time() + RETRY_BUDGET_S
    for attempt in range(MAX_RETRIES):
        try:
            r = client(EMBED_LOCATION).models.embed_content(model=EMBED_MODEL, contents=text, config=cfg)
            v = np.asarray(r.embeddings[0].values, dtype=np.float32)
            if v.shape[0] != EMBED_DIM:
                raise _Fatal(f"{v.shape[0]} dims, expected {EMBED_DIM}")
            meter.add(retries=attempt, rate_limited=n_429)
            return rid, v, ""
        except _Fatal as exc:
            last = f"Fatal: {exc}"
            break
        except Exception as exc:  # noqa: BLE001
            code = _status_code(exc)
            last = f"{type(exc).__name__} {code or ''}: {str(exc)[:120]}"
            if code == 429:
                n_429 += 1
            if code is not None and code not in RETRYABLE_CODES and 400 <= code < 500:
                break
            if attempt < MAX_RETRIES - 1:
                wait = _backoff(attempt, code, exc)
                if time.time() + wait > deadline:
                    break
                time.sleep(wait)
    meter.add(retries=MAX_RETRIES, failed=True, rate_limited=n_429)
    return rid, None, last


if not RUN_EMBEDDINGS:
    print("skipped—RUN_EMBEDDINGS is False")
elif RES is None or not QA_CLEAN:
    print("skipped—waiting for collected results with zero hard QA failures")
elif not PREFLIGHT_OK:
    print("skipped—preflight failed")
else:
    for u in sorted(x for x in ls(EMB_PREFIX) if x.endswith(".parquet")):
        d = pd.read_parquet(io.BytesIO(gcs("cat", u, binary=True).stdout))
        for rid, vec in zip(d.report_id, d.embedding):
            EMB[rid] = np.asarray(vec, dtype=np.float32)
    print(f"restored {len(EMB):,} embeddings from {EMB_PREFIX}/")
    todo = RES[~RES.report_id.isin(EMB.keys())][["report_id", "report_text"]].values.tolist()
    shard_no = len([u for u in ls(EMB_PREFIX) if u.endswith(".parquet")])
    t0 = time.time()
    errors = []
    emeter, ethrottle = Meter(), Throttle(EMBED_WORKERS)
    for start in range(0, len(todo), EMBED_SHARD):
        chunk = todo[start:start + EMBED_SHARD]
        got = []
        with ThreadPoolExecutor(max_workers=ethrottle.workers) as pool:
            for rid, vec, err in pool.map(lambda x: embed_one(x[0], x[1], emeter), chunk):
                if vec is None:
                    errors.append((rid, err))
                else:
                    got.append((rid, vec)); EMB[rid] = vec
        if got:
            sp = BASE / "emb" / f"shard-{shard_no:04d}.parquet"
            pd.DataFrame({"report_id": [g_[0] for g_ in got], "embedding": [g_[1] for g_ in got]}).to_parquet(sp, index=False)
            put(sp, f"{EMB_PREFIX}/shard-{shard_no:04d}.parquet")
            shard_no += 1
        ethrottle.adjust(emeter)
        rate = (start + len(chunk)) / max(time.time() - t0, 1)
        print(f"  {len(EMB):,}/{len(RES):,} embedded ({rate:,.1f}/s, workers {ethrottle.workers}, "
              f"429s {emeter.rate_limited}, {len(errors)} errors)")
    if errors:
        print(f"⚠️  {len(errors)} embeddings failed; rerun to retry them. First: {errors[0]}")
EMB_DONE = RES is not None and len(EMB) == len(RES)
hb(S, f"embedded {len(EMB):,} of {len(RES) if RES is not None else '-'}; complete={EMB_DONE}")

---
## 12 · Stage the corpus

Runs when every report is generated, QA-clean and embedded. Writes, to
`{workshop prefix}/staged/`:

- `scout_reports.csv.gz`—the whole corpus, headerless, the Part 1 conventions (NULL as an
  unquoted empty field, integers never float-formatted, vectors as pgvector literals at 6
  significant digits)
- `scout_reports_<season>.csv.gz`—one file per season, so a lab can load any season range
- `scout_reports.sql`—the table DDL, plus the indexes to build **after** the load
- `scout_reports_manifest.json`—column order, rows, bytes and sha256 per file

Every file is read back and checked before upload. Once staged, later runs leave the files alone
unless `FORCE_RESTAGE = True`.

In [ ]:
S = "12 · Stage the corpus"
STAGE_PREFIX = f"{GCS_WORK}/staged"
COLS = ["report_id", "player_id", "season", "player_club_id", "scout_club_id", "perspective", "report_date",
        "recommendation", "headline", "report_text", "report_embedding"]
DDL = f'''-- CymbalGoal AlloyDB workshop—scout_reports. Generated by 01-generate-scout-reports.ipynb ({PROMPT_VERSION}).
-- Apply after the CymbalGoal schema (players, clubs). Load data, THEN create the indexes below.
CREATE TABLE scout_reports (
    report_id          TEXT        PRIMARY KEY,
    player_id          INTEGER     NOT NULL REFERENCES players (player_id),
    season             INTEGER     NOT NULL,
    player_club_id     INTEGER     REFERENCES clubs (club_id),
    scout_club_id      INTEGER     NOT NULL REFERENCES clubs (club_id),
    perspective        TEXT        NOT NULL,
    report_date        DATE        NOT NULL,
    recommendation     TEXT        NOT NULL,
    headline           TEXT        NOT NULL,
    report_text        TEXT        NOT NULL,
    report_embedding   VECTOR({EMBED_DIM})
);
COMMENT ON TABLE scout_reports IS 'One scouting report per scouting club per player-season (players with 900+ minutes in covered matches). Written as of the end of that season: a report never mentions anything that happened later.';
COMMENT ON COLUMN scout_reports.season IS 'Starting year of the season. The 2024/25 season is 2024.';
COMMENT ON COLUMN scout_reports.player_club_id IS 'The club he made most appearances for that season.';
COMMENT ON COLUMN scout_reports.scout_club_id IS 'The club whose scout wrote the report. Never a club he played for that season.';
COMMENT ON COLUMN scout_reports.perspective IS 'later_signer: a club he joined after this season began (the report itself does not know that). european_opponent: a club he faced in the Champions League or Europa League that season. same_league: another club in his league.';
COMMENT ON COLUMN scout_reports.recommendation IS 'SIGN, MONITOR or PASS.';
COMMENT ON COLUMN scout_reports.report_embedding IS '{EMBED_MODEL}, {EMBED_DIM} dimensions, RETRIEVAL_DOCUMENT, embedding of report_text.';

-- ---- After the load -------------------------------------------------------------------
-- CREATE INDEX idx_scout_reports_player ON scout_reports (player_id);
-- CREATE INDEX idx_scout_reports_season ON scout_reports (season);
-- CREATE INDEX scout_reports_embedding_scann_idx ON scout_reports
--     USING scann (report_embedding cosine) WITH (num_leaves = <floor(sqrt(rows))>, quantizer = 'sq8');
'''


def pgvec(v):
    return "[" + ",".join(f"{float(x):.{EMBED_SIG_DIGITS}g}" for x in v) + "]"


STAGED = {}
_prev = json.loads(get_text(f"{STAGE_PREFIX}/scout_reports_manifest.json") or "null")
if (_prev and not FORCE_RESTAGE and _prev.get("prompt_version") == PROMPT_VERSION and RES is not None
        and _prev["files"].get("scout_reports.csv.gz", {}).get("rows") == len(RES)):
    STAGED = {k: (v["rows"], v["bytes"]) for k, v in _prev["files"].items()}
    print(f"already staged on {_prev['generated_utc']}—not rewriting (FORCE_RESTAGE = True to redo)")
    for k, v in STAGED.items():
        print(f"  {k:<28}{v[0]:>8,} rows  {v[1] / 1e6:>8.1f} MB")
elif not STAGE_EXPORT:
    print("skipped—STAGE_EXPORT is False")
elif not (RES is not None and QA_CLEAN and EMB_DONE):
    print("skipped—needs collected, QA-clean, fully embedded results")
else:
    D = R[["report_id", "player_id", "season", "player_club_id", "scout_club_id", "perspective", "report_date"]] \
          .merge(RES[["report_id", "recommendation", "headline", "report_text"]], on="report_id", validate="one_to_one")
    for c in ("player_id", "season", "player_club_id", "scout_club_id"):
        D[c] = pd.to_numeric(D[c], errors="coerce").astype("Int64")
    D["report_date"] = pd.to_datetime(D.report_date).dt.strftime("%Y-%m-%d")
    D["report_embedding"] = D.report_id.map(lambda r: pgvec(EMB[r]))
    D = D[COLS].sort_values(["season", "player_id", "report_id"])
    out = BASE / "export"
    files = {"scout_reports.csv.gz": D}
    for s, d in D.groupby("season"):
        files[f"scout_reports_{int(s)}.csv.gz"] = d
    manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "prompt_version": PROMPT_VERSION, "model": GEN_MODEL, "embedding": f"{EMBED_MODEL}@{EMBED_DIM}",
                "snapshot": {"date": SNAPSHOT_DATE, "sha256": SNAPSHOT_SHA256},
                "column_order": COLS, "header": False, "files": {}}
    for name, d in files.items():
        p = out / name
        d.to_csv(p, index=False, header=False, na_rep="", quoting=csv.QUOTE_MINIMAL, encoding="utf-8",
                 compression="gzip", lineterminator="\n")
        n, bad = 0, 0
        with gzip.open(p, "rt", encoding="utf-8", newline="") as fh:
            for row in csv.reader(fh):
                n += 1
                bad += len(row) != len(COLS)
        if n != len(d) or bad:
            raise AssertionError(f"{name}: read back {n:,} rows ({bad} bad) vs {len(d):,}")
        manifest["files"][name] = {"rows": int(len(d)), "bytes": p.stat().st_size,
                                   "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
        put(p, f"{STAGE_PREFIX}/{name}")
        STAGED[name] = (len(d), p.stat().st_size)
        print(f"  ✅ {name:<28}{len(d):>8,} rows  {p.stat().st_size / 1e6:>8.1f} MB")
    (out / "scout_reports.sql").write_text(DDL)
    put(out / "scout_reports.sql", f"{STAGE_PREFIX}/scout_reports.sql")
    (out / "scout_reports_manifest.json").write_text(json.dumps(manifest, indent=2))
    put(out / "scout_reports_manifest.json", f"{STAGE_PREFIX}/scout_reports_manifest.json")
    print(f"\n✅ STAGED to {STAGE_PREFIX}/")
hb(S, ("STAGED " + "; ".join(f"{k} {v[0]:,} rows {v[1] / 1e6:,.1f} MB" for k, v in STAGED.items()))
      if STAGED else "not staged yet")

---
## 13 · Where things stand

A one-line status, so it's obvious which run comes next.

In [ ]:
if not STAGED and not PREFLIGHT_OK:     # checked first: nothing runs until preflight passes
    NEXT = ("Preflight failed—" + ("this identity can't write to the workshop prefix: run the grant command "
            "in section 2 as the bucket owner, then Run all." if not WRITE_OK else "see section 2, then Run all."))
elif STAGED:
    NEXT = "DONE—corpus staged. Next: the load test."
elif RES is not None and QA_CLEAN and not EMB_DONE:
    NEXT = "Embeddings incomplete—rerun to continue from the saved shards."
elif RES is not None and not QA_CLEAN:
    NEXT = "Hard QA failures remain after repair—send the handback."
elif GEN and not GEN_COMPLETE:
    NEXT = (f"Generation {sum(1 for o in GEN.values() if not o.get('error')):,} of {len(R):,}—"
            "Run all again to resume from the checkpoint.")
elif SAMPLE:
    NEXT = "Sample ready—review it, then set RUN_FULL = True and Run all."
elif not WRITE_OK:
    NEXT = "Preflight could not write to the workshop prefix—run the grant command in section 2, then Run all."
else:
    NEXT = "Nothing generated yet—check the preflight."
print(NEXT)
hb("13 · Status", NEXT)

---
## 14 · Handback

One box, everything Claude needs from this run, with a **Copy** button. If your browser blocks
the button, the text is selected for you; copy it with **Edit → Copy**. The same text is saved
to `/content/cymbalgoal-scout-reports/handback.txt`.

In [ ]:
import html as _html
from IPython.display import HTML, display

lines = ["=" * 70, f"CYMBALGOAL SCOUT REPORTS—01 HANDBACK ({PROMPT_VERSION})",
         f"run {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "=" * 70]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
if SAMPLE and not GEN:            # full sample only until the full run starts
    lines += ["", "## Sample reports (facts, then output)"]
    for x in SAMPLE:
        lines += ["", f"--- {x['report_id']} | {x['player']} {season_label(x['season'])} | {x['perspective']} "
                      f"by {x['scout']} | QA hard={x['qa_hard'] or '-'} soft={x['qa_soft'] or '-'}",
                  x["facts"], f">> [{x.get('recommendation')}] {x.get('headline')}",
                  f">> {x.get('report_text') or x.get('error')}"]
writes = [c for c in GCS_LOG if c.split()[2] == "cp"]
bad = [c for c in writes if not _under(c.split()[-1], GCS_WORK)]
lines += ["", "## Bucket audit",
          f"calls {len(GCS_LOG)} ({dict(Counter(c.split()[2] for c in GCS_LOG))}); "
          f"writes {len(writes)}, all under the workshop prefix: {not bad}",
          "timings: " + ", ".join(f"{k} {v:.0f}s" for k, v in TIMERS.items()), "=" * 70]
TEXT = "\n".join(lines)
(BASE / "handback.txt").write_text(TEXT)
assert not bad, f"a write left the workshop prefix: {bad}"

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:100%">
  <button id="cg-copy" style="font-size:15px;font-weight:700;padding:10px 20px;border-radius:8px;border:0;
      background:#1f8a6d;color:#fff;cursor:pointer">Copy handback</button>
  <span id="cg-msg" style="margin-left:12px;font-weight:600;color:#1f8a6d"></span>
  <textarea id="cg-box" readonly style="display:block;width:100%;height:340px;margin-top:10px;
      font:12px/1.4 ui-monospace,Menlo,monospace;padding:10px;box-sizing:border-box">{_html.escape(TEXT)}</textarea>
</div>
<script>
(function() {{
  const b = document.getElementById("cg-copy"), t = document.getElementById("cg-box"), m = document.getElementById("cg-msg");
  b.addEventListener("click", async () => {{
    try {{ await navigator.clipboard.writeText(t.value); m.textContent = "Copied—paste it into the chat."; }}
    catch (e) {{ t.focus(); t.select(); m.textContent = "Clipboard blocked: the text is selected, use Edit → Copy."; }}
  }});
}})();
</script>'''))
print(f"{len(TEXT):,} characters; also saved to {BASE / 'handback.txt'}")